<a href="https://colab.research.google.com/github/mahashri142010/maha-ai/blob/main/MeaningGuard_AI_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Welcome to MeaningGuard!")

Welcome to MeaningGuard!


# 1. MeaningGuard — Project Overview


# MeaningGuard
### A Hybrid AI System for Detecting Meaning Changes in Multilingual Translations

**Track:** AI + Languages  
**Languages:** English, Tamil and Hindi

## Project Overview

MeaningGuard is a multilingual translation-verification prototype designed to identify potential meaning changes between original messages and their translated versions.

Unlike a conventional translation application, MeaningGuard focuses on checking whether important information survives translation.

### Core capabilities
- Multilingual semantic analysis using a pretrained transformer.
- Critical-detail checks for numbers, times and negation.
- Source-grounded verification of important requirements.
- Evidence-based decisions with human-review recommendations.

### Technology
- Python
- Google Colab
- PyTorch
- Hugging Face Transformers
- mDeBERTa multilingual NLI model
- Pandas and Scikit-learn

**Important:** MeaningGuard is a verification aid, not a replacement for professional translators or human review.


In [ ]:
# MeaningGuard - Project Setup

PROJECT_NAME = "MeaningGuard"
LANGUAGES = ["English", "Tamil", "Hindi"]

print("Project:", PROJECT_NAME)
print("Supported languages:", LANGUAGES)
print("Development environment ready!")

Project: MeaningGuard
Supported languages: ['English', 'Tamil', 'Hindi']
Development environment ready!


### Initial Dataset and Early Experiments

This section introduces the initial multilingual dataset and records the early experiments used to develop MeaningGuard.

In [ ]:

import pandas as pd

# Our first original dataset
data = [
    {
        "original_text": "The library closes at 6 PM.",
        "translated_text": "நூலகம் மாலை 6 மணிக்கு மூடப்படும்.",
        "target_language": "Tamil",
        "label": 0,
        "change_type": "None"
    },
    {
        "original_text": "The library closes at 6 PM.",
        "translated_text": "நூலகம் மாலை 8 மணிக்கு மூடப்படும்.",
        "target_language": "Tamil",
        "label": 1,
        "change_type": "Time"
    },
    {
        "original_text": "Do not use your phone during the exam.",
        "translated_text": "தேர்வின் போது உங்கள் கைப்பேசியைப் பயன்படுத்த வேண்டாம்.",
        "target_language": "Tamil",
        "label": 0,
        "change_type": "None"
    },
    {
        "original_text": "Do not use your phone during the exam.",
        "translated_text": "தேர்வின் போது உங்கள் கைப்பேசியைப் பயன்படுத்தலாம்.",
        "target_language": "Tamil",
        "label": 1,
        "change_type": "Negation"
    },
    {
        "original_text": "The library closes at 6 PM.",
        "translated_text": "पुस्तकालय शाम 6 बजे बंद होता है।",
        "target_language": "Hindi",
        "label": 0,
        "change_type": "None"
    },
    {
        "original_text": "The library closes at 6 PM.",
        "translated_text": "पुस्तकालय शाम 8 बजे बंद होता है।",
        "target_language": "Hindi",
        "label": 1,
        "change_type": "Time"
    },
    {
        "original_text": "Do not use your phone during the exam.",
        "translated_text": "परीक्षा के दौरान अपने फ़ोन का उपयोग न करें।",
        "target_language": "Hindi",
        "label": 0,
        "change_type": "None"
    },
    {
        "original_text": "Do not use your phone during the exam.",
        "translated_text": "परीक्षा के दौरान अपने फ़ोन का उपयोग कर सकते हैं।",
        "target_language": "Hindi",
        "label": 1,
        "change_type": "Negation"
    }
]

# Convert our data into a table
df = pd.DataFrame(data)

# Display the dataset
display(df)


,original_text,translated_text,target_language,label,change_type
0,The library closes at 6 PM.,நூலகம் மாலை 6 மணிக்கு மூடப்படும்.,Tamil,0,None
1,The library closes at 6 PM.,நூலகம் மாலை 8 மணிக்கு மூடப்படும்.,Tamil,1,Time
2,Do not use your phone during the exam.,தேர்வின் போது உங்கள் கைப்பேசியைப் பயன்படுத்த வ...,Tamil,0,None
3,Do not use your phone during the exam.,தேர்வின் போது உங்கள் கைப்பேசியைப் பயன்படுத்தலாம்.,Tamil,1,Negation
4,The library closes at 6 PM.,पुस्तकालय शाम 6 बजे बंद होता है।,Hindi,0,None
5,The library closes at 6 PM.,पुस्तकालय शाम 8 बजे बंद होता है।,Hindi,1,Time
6,Do not use your phone during the exam.,परीक्षा के दौरान अपने फ़ोन का उपयोग न करें।,Hindi,0,None
7,Do not use your phone during the exam.,परीक्षा के दौरान अपने फ़ोन का उपयोग कर सकते हैं।,Hindi,1,Negation


In [ ]:
# Save our dataset as a CSV file
df.to_csv("meaningguard_dataset.csv", index=False)

print("Dataset saved successfully!")

Dataset saved successfully!


# 2. Environment Setup and Dependencies

In [ ]:
!pip install -q transformers sentencepiece accelerate

# 3. Multilingual Transformer Model

In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Select our pretrained multilingual model
MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# Check available computing hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Computing device:", device)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the pretrained AI model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Move the model to the available device
model.to(device)

# Set the model to evaluation mode
model.eval()

print("MeaningGuard AI model loaded successfully!")

# Inspect the model's classification labels
print("Model labels:", model.config.id2label)


Computing device: cuda


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

MeaningGuard AI model loaded successfully!
Model labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


# 4. Natural Language Inference (NLI)

In [ ]:

def predict_relationship(premise, hypothesis):

    # Convert both sentences into model-readable tokens
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    # Send input tensors to the GPU
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Run AI inference without calculating training gradients
    with torch.inference_mode():
        outputs = model(**inputs)

    # Convert raw model scores into probabilities
    probabilities = torch.softmax(outputs.logits, dim=-1)[0]

    # Find the most likely relationship
    predicted_id = torch.argmax(probabilities).item()

    return {
        "prediction": model.config.id2label[predicted_id],
        "probabilities": {
            model.config.id2label[i]: probabilities[i].item()
            for i in range(len(probabilities))
        }
    }


def analyze_bidirectionally(original, translated):

    # Direction A: Original -> Translation
    forward = predict_relationship(original, translated)

    # Direction B: Translation -> Original
    backward = predict_relationship(translated, original)

    return {
        "forward": forward,
        "backward": backward
    }


print("MeaningGuard inference engine created!")


MeaningGuard inference engine created!


In [ ]:

original = "The library closes at 6 PM."

translated = "நூலகம் மாலை 8 மணிக்கு மூடப்படும்."

result = analyze_bidirectionally(original, translated)

print("ORIGINAL:", original)
print("TRANSLATION:", translated)

print("\nFORWARD ANALYSIS")
print(result["forward"])

print("\nBACKWARD ANALYSIS")
print(result["backward"])



ORIGINAL: The library closes at 6 PM.
TRANSLATION: நூலகம் மாலை 8 மணிக்கு மூடப்படும்.

FORWARD ANALYSIS
{'prediction': 'contradiction', 'probabilities': {'entailment': 0.393798828125, 'neutral': 0.0097503662109375, 'contradiction': 0.59619140625}}

BACKWARD ANALYSIS
{'prediction': 'contradiction', 'probabilities': {'entailment': 0.07598876953125, 'neutral': 0.00812530517578125, 'contradiction': 0.916015625}}


# 5. Critical Detail Detection

In [ ]:

import re

# Language-specific negation indicators
NEGATION_WORDS = {
    "English": ["not", "never", "no", "don't", "cannot"],
    "Tamil": ["வேண்டாம்", "இல்லை", "முடியாது", "கூடாது"],
    "Hindi": ["नहीं", "मत", "न करें", "नहीं करना", "नहीं है"]
}


def extract_numbers(text):
    # Extract numerical values from a sentence
    return re.findall(r"\d+(?:[:.]\d+)?", text)


def detect_negation(text, language):
    # Check whether known negation expressions occur
    words = NEGATION_WORDS[language]

    return any(word in text.lower() for word in words)


def detect_critical_changes(original, translated, target_language):

    # Check numerical differences
    original_numbers = extract_numbers(original)
    translated_numbers = extract_numbers(translated)

    number_changed = original_numbers != translated_numbers

    # Check negation differences
    original_negation = detect_negation(original, "English")
    translated_negation = detect_negation(
        translated, target_language
    )

    negation_changed = original_negation != translated_negation

    return {
        "original_numbers": original_numbers,
        "translated_numbers": translated_numbers,
        "number_changed": number_changed,
        "original_has_negation": original_negation,
        "translation_has_negation": translated_negation,
        "negation_changed": negation_changed
    }


print("Critical Detail Detection Engine created!")


Critical Detail Detection Engine created!


In [ ]:

# Test 1: Correct Tamil translation

original = "The library closes at 6 PM."
translated = "நூலகம் மாலை 6 மணிக்கு மூடப்படும்."

result = detect_critical_changes(
    original, translated, "Tamil"
)

print("TEST 1: PRESERVED")
print(result)


# Test 2: Changed Tamil time

original = "The library closes at 6 PM."
translated = "நூலகம் மாலை 8 மணிக்கு மூடப்படும்."

result = detect_critical_changes(
    original, translated, "Tamil"
)

print("\nTEST 2: CHANGED TIME")
print(result)


# Test 3: Negation change

original = "Do not use your phone during the exam."
translated = "தேர்வின் போது உங்கள் கைப்பேசியைப் பயன்படுத்தலாம்."

result = detect_critical_changes(
    original, translated, "Tamil"
)

print("\nTEST 3: NEGATION CHANGE")
print(result)


TEST 1: PRESERVED
{'original_numbers': ['6'], 'translated_numbers': ['6'], 'number_changed': False, 'original_has_negation': False, 'translation_has_negation': False, 'negation_changed': False}

TEST 2: CHANGED TIME
{'original_numbers': ['6'], 'translated_numbers': ['8'], 'number_changed': True, 'original_has_negation': False, 'translation_has_negation': False, 'negation_changed': False}

TEST 3: NEGATION CHANGE
{'original_numbers': [], 'translated_numbers': [], 'number_changed': False, 'original_has_negation': True, 'translation_has_negation': False, 'negation_changed': True}


In [ ]:

def meaningguard_analyze(original, translated, target_language):

    # Component 1: Bidirectional transformer analysis
    nli_result = analyze_bidirectionally(original, translated)

    forward = nli_result["forward"]["probabilities"]
    backward = nli_result["backward"]["probabilities"]

    # Component 2: Critical detail analysis
    detail_result = detect_critical_changes(
        original, translated, target_language
    )

    # Collect evidence of possible changes
    reasons = []

    if detail_result["number_changed"]:
        reasons.append("Numbers or numerical details differ")

    if detail_result["negation_changed"]:
        reasons.append("Possible negation or instruction change")

    if (
        forward["contradiction"] >= 0.65
        or backward["contradiction"] >= 0.65
    ):
        reasons.append("Transformer detected a contradiction")

    # Combine the evidence into a decision
    if detail_result["number_changed"] or detail_result["negation_changed"]:
        decision = "POSSIBLE MEANING CHANGE"

    elif (
        forward["contradiction"] >= 0.65
        or backward["contradiction"] >= 0.65
    ):
        decision = "POSSIBLE MEANING CHANGE"

    elif (
        forward["entailment"] >= 0.75
        and backward["entailment"] >= 0.75
    ):
        decision = "LIKELY MEANING PRESERVED"

    else:
        decision = "NEEDS HUMAN REVIEW"

    # Return the complete analysis
    return {
        "decision": decision,
        "reasons": reasons,
        "transformer_analysis": nli_result,
        "critical_detail_analysis": detail_result
    }


print("MeaningGuard Hybrid Decision Engine is ready!")


MeaningGuard Hybrid Decision Engine is ready!


In [ ]:

# Experiment 1: Meaning preserved

original = "The library closes at 6 PM."
translated = "நூலகம் மாலை 6 மணிக்கு மூடப்படும்."

result_1 = meaningguard_analyze(
    original, translated, "Tamil"
)

print("EXPERIMENT 1")
print("Decision:", result_1["decision"])
print("Reasons:", result_1["reasons"])


# Experiment 2: Time changed

original = "The library closes at 6 PM."
translated = "நூலகம் மாலை 8 மணிக்கு மூடப்படும்."

result_2 = meaningguard_analyze(
    original, translated, "Tamil"
)

print("\nEXPERIMENT 2")
print("Decision:", result_2["decision"])
print("Reasons:", result_2["reasons"])


# Experiment 3: Negation changed

original = "Do not use your phone during the exam."
translated = "தேர்வின் போது உங்கள் கைப்பேசியைப் பயன்படுத்தலாம்."

result_3 = meaningguard_analyze(
    original, translated, "Tamil"
)

print("\nEXPERIMENT 3")
print("Decision:", result_3["decision"])
print("Reasons:", result_3["reasons"])


EXPERIMENT 1
Decision: LIKELY MEANING PRESERVED
Reasons: []

EXPERIMENT 2
Decision: POSSIBLE MEANING CHANGE
Reasons: ['Numbers or numerical details differ', 'Transformer detected a contradiction']

EXPERIMENT 3
Decision: POSSIBLE MEANING CHANGE
Reasons: ['Possible negation or instruction change', 'Transformer detected a contradiction']


In [ ]:

def detect_omission_candidate(original, translated):

    # Compare original -> translation
    forward = predict_relationship(
        original, translated
    )["probabilities"]

    # Compare translation -> original
    backward = predict_relationship(
        translated, original
    )["probabilities"]

    # Look for asymmetric entailment
    forward_entailment = forward["entailment"]
    backward_entailment = backward["entailment"]

    difference = forward_entailment - backward_entailment

    if (
        forward_entailment >= 0.70
        and backward_entailment < 0.50
        and difference >= 0.30
    ):
        status = "POSSIBLE MISSING INFORMATION"

    else:
        status = "NO CLEAR OMISSION SIGNAL"

    return {
        "status": status,
        "forward_entailment": round(forward_entailment, 4),
        "backward_entailment": round(backward_entailment, 4),
        "asymmetry_score": round(difference, 4)
    }


print("Omission detection experiment ready!")


Omission detection experiment ready!


In [ ]:

original = "The library closes at 6 PM."

translated = "நூலகம் மாலை 6 மணிக்கு மூடப்படும்."

print(
    detect_omission_candidate(original, translated)
)


{'status': 'NO CLEAR OMISSION SIGNAL', 'forward_entailment': 0.9985, 'backward_entailment': 0.9976, 'asymmetry_score': 0.001}


In [ ]:

# Original English sentence
original = (
    "Students must bring their ID cards "
    "and admission tickets."
)

# Incomplete English version
incomplete = (
    "Students must bring their ID cards."
)

# Test the same omission using English only
english_test = analyze_bidirectionally(
    original, incomplete
)

print("FORWARD ANALYSIS")
print(english_test["forward"])

print("\nBACKWARD ANALYSIS")
print(english_test["backward"])


FORWARD ANALYSIS
{'prediction': 'entailment', 'probabilities': {'entailment': 0.9970703125, 'neutral': 0.00274658203125, 'contradiction': 0.0002911090850830078}}

BACKWARD ANALYSIS
{'prediction': 'neutral', 'probabilities': {'entailment': 0.0004887580871582031, 'neutral': 0.9990234375, 'contradiction': 0.0004553794860839844}}


In [ ]:

original = "The library closes at 6 PM."

translated = "நூலகம் மாலை 6 மணிக்கு மூடப்படும்."

print(
    detect_omission_candidate(original, translated)
)


{'status': 'NO CLEAR OMISSION SIGNAL', 'forward_entailment': 0.9985, 'backward_entailment': 0.9976, 'asymmetry_score': 0.001}


In [ ]:

def verify_atomic_facts(translated, atomic_facts):

    verification_results = []

    for fact in atomic_facts:

        # Check whether the translation supports this fact
        result = predict_relationship(
            translated,
            fact
        )

        probabilities = result["probabilities"]

        # Determine the initial evidence-based status
        if probabilities["entailment"] >= 0.75:
            status = "SUPPORTED"

        elif probabilities["neutral"] >= 0.60:
            status = "NOT ESTABLISHED"

        elif probabilities["contradiction"] >= 0.60:
            status = "POSSIBLE CONTRADICTION"

        else:
            status = "UNCERTAIN"

        verification_results.append({
            "fact": fact,
            "status": status,
            "probabilities": probabilities
        })

    return verification_results


print("Atomic Fact Verification Engine created!")


Atomic Fact Verification Engine created!


In [ ]:

translated = (
    "மாணவர்கள் தங்கள் அடையாள அட்டைகளை "
    "கொண்டு வர வேண்டும்."
)

atomic_facts = [
    "Students must bring their ID cards.",
    "Students must bring their admission tickets."
]

results = verify_atomic_facts(
    translated,
    atomic_facts
)

for item in results:

    print("\nFACT:", item["fact"])
    print("STATUS:", item["status"])
    print("PROBABILITIES:", item["probabilities"])



FACT: Students must bring their ID cards.
STATUS: SUPPORTED
PROBABILITIES: {'entailment': 0.998046875, 'neutral': 0.0017271041870117188, 'contradiction': 0.00012803077697753906}

FACT: Students must bring their admission tickets.
STATUS: SUPPORTED
PROBABILITIES: {'entailment': 0.9873046875, 'neutral': 0.012542724609375, 'contradiction': 0.00028204917907714844}


In [ ]:

original = (
    "Students must bring their ID cards "
    "and admission tickets."
)

atomic_facts = [
    "Students must bring their ID cards.",
    "Students must bring their admission tickets."
]

test_cases = {
    "Complete": (
        "மாணவர்கள் தங்கள் அடையாள அட்டைகளையும் "
        "சேர்க்கைச் சீட்டுகளையும் கொண்டு வர வேண்டும்."
    ),

    "ID card only": (
        "மாணவர்கள் தங்கள் அடையாள அட்டைகளை "
        "கொண்டு வர வேண்டும்."
    ),

    "Admission ticket only": (
        "மாணவர்கள் தங்கள் சேர்க்கைச் சீட்டுகளை "
        "கொண்டு வர வேண்டும்."
    )
}

for case_name, translated in test_cases.items():

    print("\n" + "=" * 45)
    print("TEST CASE:", case_name)

    results = verify_atomic_facts(
        translated,
        atomic_facts
    )

    for item in results:
        print("\nFACT:", item["fact"])
        print("STATUS:", item["status"])
        print(
            "ENTAILMENT:",
            round(item["probabilities"]["entailment"], 4)
        )



TEST CASE: Complete

FACT: Students must bring their ID cards.
STATUS: SUPPORTED
ENTAILMENT: 0.998

FACT: Students must bring their admission tickets.
STATUS: SUPPORTED
ENTAILMENT: 0.9985

TEST CASE: ID card only

FACT: Students must bring their ID cards.
STATUS: SUPPORTED
ENTAILMENT: 0.998

FACT: Students must bring their admission tickets.
STATUS: SUPPORTED
ENTAILMENT: 0.9873

TEST CASE: Admission ticket only

FACT: Students must bring their ID cards.
STATUS: SUPPORTED
ENTAILMENT: 0.9771

FACT: Students must bring their admission tickets.
STATUS: SUPPORTED
ENTAILMENT: 0.9985


In [ ]:

import unicodedata

TAMIL_FACT_PATTERNS = {
    "ID card requirement": [
        "அடையாள அட்டைகளை",
        "அடையாள அட்டைகளையும்"
    ],

    "Admission ticket requirement": [
        "சேர்க்கைச் சீட்டுகளை",
        "சேர்க்கைச் சீட்டுகளையும்"
    ]
}


def normalize_text(text):
    text = unicodedata.normalize("NFC", text)
    return " ".join(text.casefold().split())


print("Tamil fact dictionary created!")


Tamil fact dictionary created!


In [ ]:

def check_tamil_facts(translated, required_facts):

    normalized_translation = normalize_text(translated)

    results = []

    for fact in required_facts:

        patterns = TAMIL_FACT_PATTERNS[fact]

        matched = any(
            normalize_text(pattern) in normalized_translation
            for pattern in patterns
        )

        if matched:
            status = "PHRASE FOUND"
        else:
            status = "PHRASE NOT FOUND - HUMAN REVIEW"

        results.append({
            "fact": fact,
            "status": status
        })

    return results


print("Controlled Tamil fact checker created!")


Controlled Tamil fact checker created!


In [ ]:

test_cases = {
    "Complete": (
        "மாணவர்கள் தங்கள் அடையாள அட்டைகளையும் "
        "சேர்க்கைச் சீட்டுகளையும் கொண்டு வர வேண்டும்."
    ),

    "ID card only": (
        "மாணவர்கள் தங்கள் அடையாள அட்டைகளை "
        "கொண்டு வர வேண்டும்."
    ),

    "Admission ticket only": (
        "மாணவர்கள் தங்கள் சேர்க்கைச் சீட்டுகளை "
        "கொண்டு வர வேண்டும்."
    )
}

required_facts = [
    "ID card requirement",
    "Admission ticket requirement"
]

for case_name, translated in test_cases.items():

    print("\n" + "=" * 45)
    print("TEST CASE:", case_name)

    results = check_tamil_facts(
        translated,
        required_facts
    )

    for item in results:
        print(item["fact"], ":", item["status"])



TEST CASE: Complete
ID card requirement : PHRASE FOUND
Admission ticket requirement : PHRASE FOUND

TEST CASE: ID card only
ID card requirement : PHRASE FOUND
Admission ticket requirement : PHRASE NOT FOUND - HUMAN REVIEW

TEST CASE: Admission ticket only
ID card requirement : PHRASE NOT FOUND - HUMAN REVIEW
Admission ticket requirement : PHRASE FOUND


In [ ]:

def meaningguard_analyze_v2(
    original,
    translated,
    target_language,
    required_facts
):

    # 1. Run transformer analysis
    nli_result = analyze_bidirectionally(
        original, translated
    )

    # 2. Check numerical and negation changes
    detail_result = detect_critical_changes(
        original, translated, target_language
    )

    # 3. Run controlled fact verification
    if target_language == "Tamil":

        fact_results = check_tamil_facts(
            translated, required_facts
        )

    else:
        fact_results = None

    # 4. Collect evidence
    reasons = []

    if detail_result["number_changed"]:
        reasons.append("Possible numerical detail change")

    if detail_result["negation_changed"]:
        reasons.append("Possible negation change")

    if fact_results is not None:

        for item in fact_results:

            if item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW":
                reasons.append(
                    "Expected phrase not found: " + item["fact"]
                )

    # 5. Make a cautious decision
    if (
        detail_result["number_changed"]
        or detail_result["negation_changed"]
    ):
        decision = "POTENTIAL MEANING CHANGE - VERIFY"

    elif any(
        item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW"
        for item in (fact_results or [])
    ):
        decision = "HUMAN REVIEW REQUIRED"

    else:
        decision = "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"

    return {
        "decision": decision,
        "reasons": reasons,
        "transformer_analysis": nli_result,
        "critical_detail_analysis": detail_result,
        "fact_verification": fact_results
    }


print("MeaningGuard V2 is ready!")


MeaningGuard V2 is ready!


In [ ]:

original = (
    "Students must bring their ID cards "
    "and admission tickets."
)

required_facts = [
    "ID card requirement",
    "Admission ticket requirement"
]

for case_name, translated in test_cases.items():

    print("\n" + "=" * 50)
    print("TEST:", case_name)

    result = meaningguard_analyze_v2(
        original,
        translated,
        "Tamil",
        required_facts
    )

    print("FINAL DECISION:", result["decision"])
    print("REASONS:", result["reasons"])



TEST: Complete
FINAL DECISION: NO CHECKLIST ISSUE FOUND - NOT VERIFIED
REASONS: []

TEST: ID card only
FINAL DECISION: HUMAN REVIEW REQUIRED
REASONS: ['Expected phrase not found: Admission ticket requirement']

TEST: Admission ticket only
FINAL DECISION: HUMAN REVIEW REQUIRED
REASONS: ['Expected phrase not found: ID card requirement']


# 6. Source-Grounded Fact Verification

In [ ]:

LANGUAGE_FACT_PATTERNS = {

    "Tamil": {
        "ID card requirement": [
            "அடையாள அட்டைகளை",
            "அடையாள அட்டைகளையும்"
        ],

        "Admission ticket requirement": [
            "சேர்க்கைச் சீட்டுகளை",
            "சேர்க்கைச் சீட்டுகளையும்"
        ]
    },

    "Hindi": {
        "ID card requirement": [
            "पहचान पत्र",
            "आईडी कार्ड"
        ],

        "Admission ticket requirement": [
            "प्रवेश पत्र",
            "एडमिट कार्ड"
        ]
    }
}

print("Tamil and Hindi fact dictionaries created!")


Tamil and Hindi fact dictionaries created!


In [ ]:

def check_language_facts(
    translated,
    required_facts,
    target_language
):

    normalized_translation = normalize_text(translated)

    language_patterns = LANGUAGE_FACT_PATTERNS[
        target_language
    ]

    results = []

    for fact in required_facts:

        patterns = language_patterns[fact]

        matched = any(
            normalize_text(pattern) in normalized_translation
            for pattern in patterns
        )

        status = (
            "PHRASE FOUND"
            if matched
            else "PHRASE NOT FOUND - HUMAN REVIEW"
        )

        results.append({
            "fact": fact,
            "status": status
        })

    return results


print("Multilingual fact checker created!")


Multilingual fact checker created!


In [ ]:

def meaningguard_analyze_v3(
    original,
    translated,
    target_language,
    required_facts
):

    nli_result = analyze_bidirectionally(
        original, translated
    )

    detail_result = detect_critical_changes(
        original, translated, target_language
    )

    fact_results = None

    if target_language in LANGUAGE_FACT_PATTERNS:

        fact_results = check_language_facts(
            translated,
            required_facts,
            target_language
        )

    reasons = []

    if detail_result["number_changed"]:
        reasons.append("Possible numerical detail change")

    if detail_result["negation_changed"]:
        reasons.append("Possible negation change")

    if fact_results is not None:

        for item in fact_results:

            if item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW":
                reasons.append(
                    "Expected phrase not found: " + item["fact"]
                )

    if (
        detail_result["number_changed"]
        or detail_result["negation_changed"]
    ):
        decision = "POTENTIAL MEANING CHANGE - VERIFY"

    elif any(
        item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW"
        for item in (fact_results or [])
    ):
        decision = "HUMAN REVIEW REQUIRED"

    else:
        decision = "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"

    return {
        "decision": decision,
        "reasons": reasons,
        "transformer_analysis": nli_result,
        "critical_detail_analysis": detail_result,
        "fact_verification": fact_results
    }


print("MeaningGuard V3 is ready!")


MeaningGuard V3 is ready!


In [ ]:

original = (
    "Students must bring their ID cards "
    "and admission tickets."
)

required_facts = [
    "ID card requirement",
    "Admission ticket requirement"
]

hindi_test_cases = {

    "Complete": (
        "विद्यार्थियों को अपना पहचान पत्र "
        "और प्रवेश पत्र लाना अनिवार्य है।"
    ),

    "ID card only": (
        "विद्यार्थियों को अपना पहचान पत्र "
        "लाना अनिवार्य है।"
    ),

    "Admission ticket only": (
        "विद्यार्थियों को अपना प्रवेश पत्र "
        "लाना अनिवार्य है।"
    )
}

for case_name, translated in hindi_test_cases.items():

    print("\n" + "=" * 45)
    print("TEST:", case_name)

    result = meaningguard_analyze_v3(
        original,
        translated,
        "Hindi",
        required_facts
    )

    print("DECISION:", result["decision"])
    print("REASONS:", result["reasons"])



TEST: Complete
DECISION: NO CHECKLIST ISSUE FOUND - NOT VERIFIED
REASONS: []

TEST: ID card only
DECISION: HUMAN REVIEW REQUIRED
REASONS: ['Expected phrase not found: Admission ticket requirement']

TEST: Admission ticket only
DECISION: HUMAN REVIEW REQUIRED
REASONS: ['Expected phrase not found: ID card requirement']


In [ ]:

import re

def extract_times(text):

    text = normalize_text(text)
    found = []

    # English: 6 PM, 6:30 AM
    english_pattern = (
        r"\b(1[0-2]|0?[1-9])"
        r"(?::([0-5]\d))?\s*"
        r"(a\.?m\.?|p\.?m\.?)\b"
    )

    for match in re.finditer(english_pattern, text):

        hour = int(match.group(1))
        minute = int(match.group(2) or 0)
        marker = match.group(3).replace(".", "")

        if marker == "pm":
            hour = hour % 12 + 12
        else:
            hour = hour % 12

        found.append({
            "minutes": hour * 60 + minute,
            "format": "12-hour"
        })

    # Tamil and Hindi time expressions
    regional_patterns = [
        (r"(காலை|க.பொ.)\s*(\d{1,2})(?::([0-5]\d))?", "am"),
        (r"(மதியம்|மாலை|இரவு)\s*(\d{1,2})(?::([0-5]\d))?", "pm"),
        (r"(सुबह|प्रातः)\s*(\d{1,2})(?::([0-5]\d))?", "am"),
        (r"(दोपहर|शाम|रात)\s*(\d{1,2})(?::([0-5]\d))?", "pm")
    ]

    for pattern, period in regional_patterns:

        for match in re.finditer(pattern, text):

            hour = int(match.group(2))
            minute = int(match.group(3) or 0)

            if 1 <= hour <= 12:

                if period == "pm":
                    hour = hour % 12 + 12
                else:
                    hour = hour % 12

                found.append({
                    "minutes": hour * 60 + minute,
                    "format": "regional"
                })

    # 24-hour format: 18:00, 07:30
    for match in re.finditer(
        r"\b([01]?\d|2[0-3]):([0-5]\d)\b",
        text
    ):

        # Avoid extracting the time again from 6:00 PM
        if any(
            match.start() >= m.start()
            and match.end() <= m.end()
            for m in re.finditer(english_pattern, text)
        ):
            continue

        hour = int(match.group(1))
        minute = int(match.group(2))

        found.append({
            "minutes": hour * 60 + minute,
            "format": "24-hour"
        })

    return sorted(item["minutes"] for item in found)


print("Multilingual time extraction engine created!")


Multilingual time extraction engine created!


In [ ]:

time_tests = [
    "The library closes at 6 PM.",
    "The library closes at 6 AM.",
    "The library closes at 18:00.",
    "The library closes at 17:30.",
    "நூலகம் மாலை 6 மணிக்கு மூடப்படும்.",
    "நூலகம் காலை 6 மணிக்கு மூடப்படும்.",
    "पुस्तकालय शाम 6 बजे बंद होता है।",
    "पुस्तकालय सुबह 6 बजे बंद होता है।"
]

for sentence in time_tests:

    print("\nSentence:", sentence)
    print("Normalized time:", extract_times(sentence))



Sentence: The library closes at 6 PM.
Normalized time: [1080]

Sentence: The library closes at 6 AM.
Normalized time: [360]

Sentence: The library closes at 18:00.
Normalized time: [1080]

Sentence: The library closes at 17:30.
Normalized time: [1050]

Sentence: நூலகம் மாலை 6 மணிக்கு மூடப்படும்.
Normalized time: [1080]

Sentence: நூலகம் காலை 6 மணிக்கு மூடப்படும்.
Normalized time: [360]

Sentence: पुस्तकालय शाम 6 बजे बंद होता है।
Normalized time: [1080]

Sentence: पुस्तकालय सुबह 6 बजे बंद होता है।
Normalized time: [360]


In [ ]:

def compare_times(original, translated):

    original_times = extract_times(original)
    translated_times = extract_times(translated)

    if not original_times or not translated_times:
        status = "TIME COULD NOT BE VERIFIED"

    elif original_times == translated_times:
        status = "TIME PRESERVED"

    else:
        status = "POSSIBLE TIME CHANGE"

    return {
        "original_times": original_times,
        "translated_times": translated_times,
        "status": status
    }


print(compare_times(
    "The library closes at 6 PM.",
    "நூலகம் காலை 6 மணிக்கு மூடப்படும்."
))


{'original_times': [1080], 'translated_times': [360], 'status': 'POSSIBLE TIME CHANGE'}


In [ ]:

def remove_time_expressions(text):

    patterns = [
        # English AM/PM
        r"\b(1[0-2]|0?[1-9])(?::[0-5]\d)?\s*(?:a\.?m\.?|p\.?m\.?)\b",

        # 24-hour time
        r"\b(?:[01]?\d|2[0-3]):[0-5]\d\b",

        # Tamil time expressions
        r"(?:காலை|மதியம்|மாலை|இரவு)\s*\d{1,2}(?::[0-5]\d)?",

        # Hindi time expressions
        r"(?:सुबह|प्रातः|दोपहर|शाम|रात)\s*\d{1,2}(?::[0-5]\d)?"
    ]

    cleaned = text

    for pattern in patterns:
        cleaned = re.sub(pattern, " ", cleaned, flags=re.IGNORECASE)

    return cleaned


def detect_critical_changes_v2(
    original,
    translated,
    target_language
):

    # Compare recognised times
    time_result = compare_times(original, translated)

    # Remove recognised times before comparing other digits
    original_remaining = remove_time_expressions(original)
    translated_remaining = remove_time_expressions(translated)

    original_numbers = extract_numbers(original_remaining)
    translated_numbers = extract_numbers(translated_remaining)

    number_changed = original_numbers != translated_numbers

    # Existing preliminary negation check
    original_negation = detect_negation(original, "English")
    translated_negation = detect_negation(
        translated, target_language
    )

    negation_changed = (
        original_negation != translated_negation
    )

    return {
        "time_analysis": time_result,
        "number_analysis": {
            "original_numbers": original_numbers,
            "translated_numbers": translated_numbers,
            "number_changed": number_changed
        },
        "negation_analysis": {
            "original_has_negation": original_negation,
            "translation_has_negation": translated_negation,
            "negation_changed": negation_changed
        }
    }


print("Critical Detail Engine V2 created!")


Critical Detail Engine V2 created!


In [ ]:

time_comparison_tests = [

    (
        "6 PM vs 8 PM",
        "The library closes at 6 PM.",
        "The library closes at 8 PM."
    ),

    (
        "6 PM vs 6 AM",
        "The library closes at 6 PM.",
        "நூலகம் காலை 6 மணிக்கு மூடப்படும்."
    ),

    (
        "6 PM vs 18:00",
        "The library closes at 6 PM.",
        "The library closes at 18:00."
    ),

    (
        "Different time and quantity",
        "Submit 3 documents by 6 PM.",
        "Submit 5 documents by 8 PM."
    )
]

for name, original, translated in time_comparison_tests:

    print("\n" + "=" * 45)
    print("TEST:", name)

    result = detect_critical_changes_v2(
        original,
        translated,
        "Tamil" if "நூலகம்" in translated else "English"
    )

    print(result)



TEST: 6 PM vs 8 PM
{'time_analysis': {'original_times': [1080], 'translated_times': [1200], 'status': 'POSSIBLE TIME CHANGE'}, 'number_analysis': {'original_numbers': [], 'translated_numbers': [], 'number_changed': False}, 'negation_analysis': {'original_has_negation': False, 'translation_has_negation': False, 'negation_changed': False}}

TEST: 6 PM vs 6 AM
{'time_analysis': {'original_times': [1080], 'translated_times': [360], 'status': 'POSSIBLE TIME CHANGE'}, 'number_analysis': {'original_numbers': [], 'translated_numbers': [], 'number_changed': False}, 'negation_analysis': {'original_has_negation': False, 'translation_has_negation': False, 'negation_changed': False}}

TEST: 6 PM vs 18:00
{'time_analysis': {'original_times': [1080], 'translated_times': [1080], 'status': 'TIME PRESERVED'}, 'number_analysis': {'original_numbers': [], 'translated_numbers': [], 'number_changed': False}, 'negation_analysis': {'original_has_negation': False, 'translation_has_negation': False, 'negation_c

In [ ]:

def meaningguard_analyze_v4(original, translated, target_language, required_facts):
    # 1. Run the multilingual transformer analysis
    nli_result = analyze_bidirectionally(original, translated)

    # 2. Run the improved critical-detail checks
    detail_result = detect_critical_changes_v2(
        original, translated, target_language
    )

    # 3. Check required facts for Tamil or Hindi
    fact_results = None

    if target_language in LANGUAGE_FACT_PATTERNS:
        fact_results = check_language_facts(
            translated, required_facts, target_language
        )

    reasons = []

    time_status = detail_result["time_analysis"]["status"]
    number_changed = detail_result["number_analysis"]["number_changed"]
    negation_changed = detail_result["negation_analysis"]["negation_changed"]

    # Detect possible changes
    if time_status == "POSSIBLE TIME CHANGE":
        reasons.append("Possible time change detected")

    if number_changed:
        reasons.append("Possible numerical detail change detected")

    if negation_changed:
        reasons.append("Possible negation or instruction change")

    # Identify checks that need human review
    human_review_needed = False

    if time_status == "TIME COULD NOT BE VERIFIED":
        reasons.append("Time could not be verified")
        human_review_needed = True

    if fact_results is not None:
        for item in fact_results:
            if item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW":
                reasons.append("Expected phrase not found: " + item["fact"])
                human_review_needed = True

    # 4. Final cautious decision
    possible_change = (
        time_status == "POSSIBLE TIME CHANGE"
        or number_changed
        or negation_changed
    )

    if possible_change:
        decision = "POTENTIAL MEANING CHANGE - VERIFY"
    elif human_review_needed:
        decision = "HUMAN REVIEW REQUIRED"
    else:
        decision = "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"

    return {
        "decision": decision,
        "reasons": reasons,
        "transformer_analysis": nli_result,
        "critical_detail_analysis": detail_result,
        "fact_verification": fact_results
    }

print("MeaningGuard V4 is ready!")


MeaningGuard V4 is ready!


In [ ]:

original = "Students must bring their ID cards and admission tickets."

translated = "विद्यार्थियों को अपना पहचान पत्र और प्रवेश पत्र लाना अनिवार्य है।"

required_facts = [
    "ID card requirement",
    "Admission ticket requirement"
]

result = meaningguard_analyze_v4(
    original,
    translated,
    "Hindi",
    required_facts
)

print("FINAL DECISION:", result["decision"])
print("REASONS:", result["reasons"])
print("FACT VERIFICATION:", result["fact_verification"])


FINAL DECISION: HUMAN REVIEW REQUIRED
REASONS: ['Time could not be verified']
FACT VERIFICATION: [{'fact': 'ID card requirement', 'status': 'PHRASE FOUND'}, {'fact': 'Admission ticket requirement', 'status': 'PHRASE FOUND'}]


In [ ]:

def contains_time_expression(text):
    patterns = [
        r"\b(1[0-2]|0?[1-9])(?::[0-5]\d)?\s*(a\.?m\.?|p\.?m\.?)\b",
        r"\b([01]?\d|2[0-3]):[0-5]\d\b",
        r"(காலை|மதியம்|மாலை|இரவு)\s*\d{1,2}",
        r"(सुबह|प्रातः|दोपहर|शाम|रात)\s*\d{1,2}"
    ]

    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in patterns
    )


def meaningguard_analyze_v4(original, translated, target_language, required_facts):

    # 1. Transformer analysis
    nli_result = analyze_bidirectionally(original, translated)

    # 2. Critical detail analysis
    detail_result = detect_critical_changes_v2(
        original, translated, target_language
    )

    # 3. Fact verification
    fact_results = None

    if target_language in LANGUAGE_FACT_PATTERNS:
        fact_results = check_language_facts(
            translated, required_facts, target_language
        )

    reasons = []

    time_status = detail_result["time_analysis"]["status"]
    number_changed = detail_result["number_analysis"]["number_changed"]
    negation_changed = detail_result["negation_analysis"]["negation_changed"]

    # 4. Detect possible changes
    if time_status == "POSSIBLE TIME CHANGE":
        reasons.append("Possible time change detected")

    if number_changed:
        reasons.append("Possible numerical detail change detected")

    if negation_changed:
        reasons.append("Possible negation or instruction change")

    # 5. Decide whether human review is needed
    human_review_needed = False

    if (
        time_status == "TIME COULD NOT BE VERIFIED"
        and (
            contains_time_expression(original)
            or contains_time_expression(translated)
        )
    ):
        reasons.append("Time could not be verified")
        human_review_needed = True

    if fact_results is not None:
        for item in fact_results:
            if item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW":
                reasons.append("Expected phrase not found: " + item["fact"])
                human_review_needed = True

    # 6. Final decision
    possible_change = (
        time_status == "POSSIBLE TIME CHANGE"
        or number_changed
        or negation_changed
    )

    if possible_change:
        decision = "POTENTIAL MEANING CHANGE - VERIFY"

    elif human_review_needed:
        decision = "HUMAN REVIEW REQUIRED"

    else:
        decision = "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"

    return {
        "decision": decision,
        "reasons": reasons,
        "transformer_analysis": nli_result,
        "critical_detail_analysis": detail_result,
        "fact_verification": fact_results
    }


print("MeaningGuard V4 updated successfully!")


MeaningGuard V4 updated successfully!


In [ ]:

original = "Students must bring their ID cards and admission tickets."

translated = "विद्यार्थियों को अपना पहचान पत्र और प्रवेश पत्र लाना अनिवार्य है।"

required_facts = [
    "ID card requirement",
    "Admission ticket requirement"
]

result = meaningguard_analyze_v4(
    original,
    translated,
    "Hindi",
    required_facts
)

print("FINAL DECISION:", result["decision"])
print("REASONS:", result["reasons"])
print("FACT VERIFICATION:", result["fact_verification"])


FINAL DECISION: NO CHECKLIST ISSUE FOUND - NOT VERIFIED
REASONS: []
FACT VERIFICATION: [{'fact': 'ID card requirement', 'status': 'PHRASE FOUND'}, {'fact': 'Admission ticket requirement', 'status': 'PHRASE FOUND'}]


In [ ]:

original = "Students must bring their ID cards and admission tickets."

translated = "विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य है।"

required_facts = [
    "ID card requirement",
    "Admission ticket requirement"
]

result = meaningguard_analyze_v4(
    original,
    translated,
    "Hindi",
    required_facts
)

print("FINAL DECISION:", result["decision"])
print("REASONS:", result["reasons"])
print("FACT VERIFICATION:", result["fact_verification"])


FINAL DECISION: HUMAN REVIEW REQUIRED
REASONS: ['Expected phrase not found: Admission ticket requirement']
FACT VERIFICATION: [{'fact': 'ID card requirement', 'status': 'PHRASE FOUND'}, {'fact': 'Admission ticket requirement', 'status': 'PHRASE NOT FOUND - HUMAN REVIEW'}]


In [ ]:

import pandas as pd

evaluation_data = [
    {
        "id": "H01",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "विद्यार्थियों को अपना पहचान पत्र और प्रवेश पत्र लाना अनिवार्य है।",
        "target_language": "Hindi",
        "expected_label": 0,
        "case_type": "Complete translation"
    },
    {
        "id": "H02",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य है।",
        "target_language": "Hindi",
        "expected_label": 1,
        "case_type": "Admission ticket omitted"
    },
    {
        "id": "H03",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "विद्यार्थियों को अपना प्रवेश पत्र लाना अनिवार्य है।",
        "target_language": "Hindi",
        "expected_label": 1,
        "case_type": "ID card omitted"
    },
    {
        "id": "T01",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "மாணவர்கள் தங்கள் அடையாள அட்டைகளையும் சேர்க்கைச் சீட்டுகளையும் கொண்டு வர வேண்டும்.",
        "target_language": "Tamil",
        "expected_label": 0,
        "case_type": "Complete translation"
    },
    {
        "id": "T02",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "மாணவர்கள் தங்கள் அடையாள அட்டைகளை கொண்டு வர வேண்டும்.",
        "target_language": "Tamil",
        "expected_label": 1,
        "case_type": "Admission ticket omitted"
    },
    {
        "id": "T03",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "மாணவர்கள் தங்கள் சேர்க்கைச் சீட்டுகளை கொண்டு வர வேண்டும்.",
        "target_language": "Tamil",
        "expected_label": 1,
        "case_type": "ID card omitted"
    }
]

evaluation_df = pd.DataFrame(evaluation_data)

evaluation_df["validation_status"] = "Needs fluent-speaker review"

print("Evaluation dataset created!")
print("Total examples:", len(evaluation_df))
display(evaluation_df)


Evaluation dataset created!
Total examples: 6


,id,original,translated,target_language,expected_label,case_type,validation_status
0,H01,Students must bring their ID cards and admissi...,विद्यार्थियों को अपना पहचान पत्र और प्रवेश पत्...,Hindi,0,Complete translation,Needs fluent-speaker review
1,H02,Students must bring their ID cards and admissi...,विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य...,Hindi,1,Admission ticket omitted,Needs fluent-speaker review
2,H03,Students must bring their ID cards and admissi...,विद्यार्थियों को अपना प्रवेश पत्र लाना अनिवार्...,Hindi,1,ID card omitted,Needs fluent-speaker review
3,T01,Students must bring their ID cards and admissi...,மாணவர்கள் தங்கள் அடையாள அட்டைகளையும் சேர்க்கைச...,Tamil,0,Complete translation,Needs fluent-speaker review
4,T02,Students must bring their ID cards and admissi...,மாணவர்கள் தங்கள் அடையாள அட்டைகளை கொண்டு வர வேண...,Tamil,1,Admission ticket omitted,Needs fluent-speaker review
5,T03,Students must bring their ID cards and admissi...,மாணவர்கள் தங்கள் சேர்க்கைச் சீட்டுகளை கொண்டு வ...,Tamil,1,ID card omitted,Needs fluent-speaker review


In [ ]:

evaluation_results = []

for _, row in evaluation_df.iterrows():

    required_facts = [
        "ID card requirement",
        "Admission ticket requirement"
    ]

    result = meaningguard_analyze_v4(
        row["original"],
        row["translated"],
        row["target_language"],
        required_facts
    )

    # Convert MeaningGuard's decision into an evaluation label
    if result["decision"] == "NO CHECKLIST ISSUE FOUND - NOT VERIFIED":
        predicted_label = 0

    elif result["decision"] == "POTENTIAL MEANING CHANGE - VERIFY":
        predicted_label = 1

    else:
        predicted_label = 1  # Human review is treated as flagged

    evaluation_results.append({
        "id": row["id"],
        "language": row["target_language"],
        "case_type": row["case_type"],
        "expected_label": row["expected_label"],
        "predicted_label": predicted_label,
        "decision": result["decision"],
        "correct_prediction": (
            predicted_label == row["expected_label"]
        )
    })

results_df = pd.DataFrame(evaluation_results)

print("MEANINGGUARD EVALUATION RESULTS")
display(results_df)

print("Correct predictions:",
      results_df["correct_prediction"].sum(), "/", len(results_df))


MEANINGGUARD EVALUATION RESULTS


,id,language,case_type,expected_label,predicted_label,decision,correct_prediction
0,H01,Hindi,Complete translation,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
1,H02,Hindi,Admission ticket omitted,1,1,HUMAN REVIEW REQUIRED,True
2,H03,Hindi,ID card omitted,1,1,HUMAN REVIEW REQUIRED,True
3,T01,Tamil,Complete translation,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
4,T02,Tamil,Admission ticket omitted,1,1,HUMAN REVIEW REQUIRED,True
5,T03,Tamil,ID card omitted,1,1,HUMAN REVIEW REQUIRED,True


Correct predictions: 6 / 6


In [ ]:

results_df.to_csv(
    "meaningguard_initial_evaluation.csv",
    index=False
)

print("Initial evaluation saved successfully!")
print("File: meaningguard_initial_evaluation.csv")


Initial evaluation saved successfully!
File: meaningguard_initial_evaluation.csv


In [ ]:

import pandas as pd

data_v2 = [

    # ENGLISH — 10 examples
    {
        "id": "E01",
        "original": "The office opens at 9 AM.",
        "translated": "The office opens at 9 AM.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Time preserved",
        "required_facts": []
    },
    {
        "id": "E02",
        "original": "The office opens at 9 AM.",
        "translated": "The office opens at 10 AM.",
        "target_language": "English",
        "expected_label": 1,
        "case_type": "Time changed",
        "required_facts": []
    },
    {
        "id": "E03",
        "original": "Bring 3 forms.",
        "translated": "Bring 3 forms.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Number preserved",
        "required_facts": []
    },
    {
        "id": "E04",
        "original": "Bring 3 forms.",
        "translated": "Bring 5 forms.",
        "target_language": "English",
        "expected_label": 1,
        "case_type": "Number changed",
        "required_facts": []
    },
    {
        "id": "E05",
        "original": "Do not share your password.",
        "translated": "Do not share your password.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Negation preserved",
        "required_facts": []
    },
    {
        "id": "E06",
        "original": "Do not share your password.",
        "translated": "Share your password.",
        "target_language": "English",
        "expected_label": 1,
        "case_type": "Negation changed",
        "required_facts": []
    },
    {
        "id": "E07",
        "original": "Students must bring their ID cards.",
        "translated": "Students are required to bring their ID cards.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Meaning preserved paraphrase",
        "required_facts": []
    },
    {
        "id": "E08",
        "original": "The school trip begins at 7 AM.",
        "translated": "The school trip begins at 8 AM.",
        "target_language": "English",
        "expected_label": 1,
        "case_type": "Time changed",
        "required_facts": []
    },
    {
        "id": "E09",
        "original": "Submit 2 photographs.",
        "translated": "Submit 4 photographs.",
        "target_language": "English",
        "expected_label": 1,
        "case_type": "Number changed",
        "required_facts": []
    },
    {
        "id": "E10",
        "original": "Students must wear uniforms.",
        "translated": "Students must wear uniforms.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Meaning preserved",
        "required_facts": []
    },

    # HINDI — 10 examples
    {
        "id": "H01",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "विद्यार्थियों को अपना पहचान पत्र और प्रवेश पत्र लाना अनिवार्य है।",
        "target_language": "Hindi",
        "expected_label": 0,
        "case_type": "Complete translation",
        "required_facts": ["ID card requirement", "Admission ticket requirement"]
    },
    {
        "id": "H02",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य है।",
        "target_language": "Hindi",
        "expected_label": 1,
        "case_type": "Admission ticket omitted",
        "required_facts": ["ID card requirement", "Admission ticket requirement"]
    },
    {
        "id": "H03",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "विद्यार्थियों को अपना प्रवेश पत्र लाना अनिवार्य है।",
        "target_language": "Hindi",
        "expected_label": 1,
        "case_type": "ID card omitted",
        "required_facts": ["ID card requirement", "Admission ticket requirement"]
    },
    {
        "id": "H04",
        "original": "The meeting starts at 6 PM.",
        "translated": "बैठक शाम 6 बजे शुरू होगी।",
        "target_language": "Hindi",
        "expected_label": 0,
        "case_type": "Time preserved",
        "required_facts": []
    },
    {
        "id": "H05",
        "original": "The meeting starts at 6 PM.",
        "translated": "बैठक सुबह 6 बजे शुरू होगी।",
        "target_language": "Hindi",
        "expected_label": 1,
        "case_type": "Time changed",
        "required_facts": []
    },
    {
        "id": "H06",
        "original": "Bring 3 documents.",
        "translated": "3 दस्तावेज़ लाएँ।",
        "target_language": "Hindi",
        "expected_label": 0,
        "case_type": "Number preserved",
        "required_facts": []
    },
    {
        "id": "H07",
        "original": "Bring 3 documents.",
        "translated": "4 दस्तावेज़ लाएँ।",
        "target_language": "Hindi",
        "expected_label": 1,
        "case_type": "Number changed",
        "required_facts": []
    },
    {
        "id": "H08",
        "original": "Students must not share passwords.",
        "translated": "विद्यार्थियों को पासवर्ड साझा नहीं करने चाहिए।",
        "target_language": "Hindi",
        "expected_label": 0,
        "case_type": "Negation preserved",
        "required_facts": []
    },
    {
        "id": "H09",
        "original": "Students must not share passwords.",
        "translated": "विद्यार्थियों को पासवर्ड साझा करने चाहिए।",
        "target_language": "Hindi",
        "expected_label": 1,
        "case_type": "Negation changed",
        "required_facts": []
    },
    {
        "id": "H10",
        "original": "The library closes at 6 PM.",
        "translated": "पुस्तकालय शाम 6 बजे बंद होता है।",
        "target_language": "Hindi",
        "expected_label": 0,
        "case_type": "Time preserved",
        "required_facts": []
    },

    # TAMIL — 10 examples
    {
        "id": "T01",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "மாணவர்கள் தங்கள் அடையாள அட்டைகளையும் சேர்க்கைச் சீட்டுகளையும் கொண்டு வர வேண்டும்.",
        "target_language": "Tamil",
        "expected_label": 0,
        "case_type": "Complete translation",
        "required_facts": ["ID card requirement", "Admission ticket requirement"]
    },
    {
        "id": "T02",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "மாணவர்கள் தங்கள் அடையாள அட்டைகளை கொண்டு வர வேண்டும்.",
        "target_language": "Tamil",
        "expected_label": 1,
        "case_type": "Admission ticket omitted",
        "required_facts": ["ID card requirement", "Admission ticket requirement"]
    },
    {
        "id": "T03",
        "original": "Students must bring their ID cards and admission tickets.",
        "translated": "மாணவர்கள் தங்கள் சேர்க்கைச் சீட்டுகளை கொண்டு வர வேண்டும்.",
        "target_language": "Tamil",
        "expected_label": 1,
        "case_type": "ID card omitted",
        "required_facts": ["ID card requirement", "Admission ticket requirement"]
    },
    {
        "id": "T04",
        "original": "The library closes at 6 PM.",
        "translated": "நூலகம் மாலை 6 மணிக்கு மூடப்படும்.",
        "target_language": "Tamil",
        "expected_label": 0,
        "case_type": "Time preserved",
        "required_facts": []
    },
    {
        "id": "T05",
        "original": "The library closes at 6 PM.",
        "translated": "நூலகம் காலை 6 மணிக்கு மூடப்படும்.",
        "target_language": "Tamil",
        "expected_label": 1,
        "case_type": "Time changed",
        "required_facts": []
    },
    {
        "id": "T06",
        "original": "Bring 3 documents.",
        "translated": "3 ஆவணங்களை கொண்டு வாருங்கள்.",
        "target_language": "Tamil",
        "expected_label": 0,
        "case_type": "Number preserved",
        "required_facts": []
    },
    {
        "id": "T07",
        "original": "Bring 3 documents.",
        "translated": "4 ஆவணங்களை கொண்டு வாருங்கள்.",
        "target_language": "Tamil",
        "expected_label": 1,
        "case_type": "Number changed",
        "required_facts": []
    },
    {
        "id": "T08",
        "original": "Students must not share passwords.",
        "translated": "மாணவர்கள் கடவுச்சொற்களை பகிரக்கூடாது.",
        "target_language": "Tamil",
        "expected_label": 0,
        "case_type": "Negation preserved",
        "required_facts": []
    },
    {
        "id": "T09",
        "original": "Students must not share passwords.",
        "translated": "மாணவர்கள் கடவுச்சொற்களை பகிர வேண்டும்.",
        "target_language": "Tamil",
        "expected_label": 1,
        "case_type": "Negation changed",
        "required_facts": []
    },
    {
        "id": "T10",
        "original": "The meeting starts at 6 PM.",
        "translated": "கூட்டம் மாலை 6 மணிக்கு தொடங்கும்.",
        "target_language": "Tamil",
        "expected_label": 0,
        "case_type": "Time preserved",
        "required_facts": []
    }
]

evaluation_v2 = pd.DataFrame(data_v2)

evaluation_v2["validation_status"] = "Needs independent language review"

print("MeaningGuard Dataset 2.0 created!")
print("Total examples:", len(evaluation_v2))
print("\nLanguage distribution:")
print(evaluation_v2["target_language"].value_counts())
print("\nCase distribution:")
print(evaluation_v2["case_type"].value_counts())

display(evaluation_v2)


MeaningGuard Dataset 2.0 created!
Total examples: 30

Language distribution:
target_language
English    10
Hindi      10
Tamil      10
Name: count, dtype: int64

Case distribution:
case_type
Time preserved                  5
Time changed                    4
Number changed                  4
Number preserved                3
Negation preserved              3
Negation changed                3
Complete translation            2
ID card omitted                 2
Admission ticket omitted        2
Meaning preserved paraphrase    1
Meaning preserved               1
Name: count, dtype: int64


,id,original,translated,target_language,expected_label,case_type,required_facts,validation_status
0,E01,The office opens at 9 AM.,The office opens at 9 AM.,English,0,Time preserved,[],Needs independent language review
1,E02,The office opens at 9 AM.,The office opens at 10 AM.,English,1,Time changed,[],Needs independent language review
2,E03,Bring 3 forms.,Bring 3 forms.,English,0,Number preserved,[],Needs independent language review
3,E04,Bring 3 forms.,Bring 5 forms.,English,1,Number changed,[],Needs independent language review
4,E05,Do not share your password.,Do not share your password.,English,0,Negation preserved,[],Needs independent language review
5,E06,Do not share your password.,Share your password.,English,1,Negation changed,[],Needs independent language review
6,E07,Students must bring their ID cards.,Students are required to bring their ID cards.,English,0,Meaning preserved paraphrase,[],Needs independent language review
7,E08,The school trip begins at 7 AM.,The school trip begins at 8 AM.,English,1,Time changed,[],Needs independent language review
8,E09,Submit 2 photographs.,Submit 4 photographs.,English,1,Number changed,[],Needs independent language review
9,E10,Students must wear uniforms.,Students must wear uniforms.,English,0,Meaning preserved,[],Needs independent language review


In [ ]:

evaluation_v2_results = []

for _, row in evaluation_v2.iterrows():

    result = meaningguard_analyze_v4(
        original=row["original"],
        translated=row["translated"],
        target_language=row["target_language"],
        required_facts=row["required_facts"]
    )

    # Convert the system decision into a binary prediction
    if result["decision"] == "NO CHECKLIST ISSUE FOUND - NOT VERIFIED":
        predicted_label = 0
    else:
        predicted_label = 1

    evaluation_v2_results.append({
        "id": row["id"],
        "language": row["target_language"],
        "case_type": row["case_type"],
        "expected_label": row["expected_label"],
        "predicted_label": predicted_label,
        "decision": result["decision"],
        "correct_prediction": (
            predicted_label == row["expected_label"]
        )
    })

results_v2_df = pd.DataFrame(evaluation_v2_results)

print("MEANINGGUARD — DATASET 2.0 RESULTS")
display(results_v2_df)

print("\nCORRECT PREDICTIONS:",
      results_v2_df["correct_prediction"].sum(),
      "/",
      len(results_v2_df))

print("\nRESULTS BY LANGUAGE:")
print(
    results_v2_df.groupby("language")["correct_prediction"]
    .agg(["sum", "count"])
)


MEANINGGUARD — DATASET 2.0 RESULTS


,id,language,case_type,expected_label,predicted_label,decision,correct_prediction
0,E01,English,Time preserved,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
1,E02,English,Time changed,1,1,POTENTIAL MEANING CHANGE - VERIFY,True
2,E03,English,Number preserved,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
3,E04,English,Number changed,1,1,POTENTIAL MEANING CHANGE - VERIFY,True
4,E05,English,Negation preserved,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
5,E06,English,Negation changed,1,1,POTENTIAL MEANING CHANGE - VERIFY,True
6,E07,English,Meaning preserved paraphrase,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
7,E08,English,Time changed,1,1,POTENTIAL MEANING CHANGE - VERIFY,True
8,E09,English,Number changed,1,1,POTENTIAL MEANING CHANGE - VERIFY,True
9,E10,English,Meaning preserved,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True



CORRECT PREDICTIONS: 30 / 30

RESULTS BY LANGUAGE:
          sum  count
language            
English    10     10
Hindi      10     10
Tamil      10     10


In [ ]:

from google.colab import files
import os
import zipfile

# Create both CSV files
evaluation_v2.to_csv(
    "/content/meaningguard_dataset_v2.csv",
    index=False
)

results_v2_df.to_csv(
    "/content/meaningguard_dataset_v2_results.csv",
    index=False
)

# Package both files into one ZIP
with zipfile.ZipFile(
    "/content/MeaningGuard_Dataset_2.0.zip", "w"
) as zip_file:

    zip_file.write("/content/meaningguard_dataset_v2.csv")
    zip_file.write("/content/meaningguard_dataset_v2_results.csv")

print("Both CSV files created successfully!")
print("ZIP file ready for download!")

# Download the ZIP
files.download("/content/MeaningGuard_Dataset_2.0.zip")


Both CSV files created successfully!
ZIP file ready for download!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

challenge_data = [
    {
        "id": "C01",
        "original": "Bring three documents.",
        "translated": "Bring 3 documents.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Written number equivalent",
        "required_facts": []
    },
    {
        "id": "C02",
        "original": "The event begins at 6 PM.",
        "translated": "The event begins at 18:00.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Equivalent time format",
        "required_facts": []
    },
    {
        "id": "C03",
        "original": "Do not enter the room.",
        "translated": "You are not allowed to enter the room.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Negation paraphrase",
        "required_facts": []
    },
    {
        "id": "C04",
        "original": "Bring three documents.",
        "translated": "Bring four documents.",
        "target_language": "English",
        "expected_label": 1,
        "case_type": "Written number changed",
        "required_facts": []
    },
    {
        "id": "C05",
        "original": "The meeting is at 6 PM.",
        "translated": "The meeting is at 6 AM.",
        "target_language": "English",
        "expected_label": 1,
        "case_type": "AM PM change",
        "required_facts": []
    },
    {
        "id": "C06",
        "original": "Students must bring their identity cards.",
        "translated": "विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य है।",
        "target_language": "Hindi",
        "expected_label": 0,
        "case_type": "Alternative Hindi wording",
        "required_facts": ["ID card requirement"]
    },
    {
        "id": "C07",
        "original": "Students must bring their identity cards and admission tickets.",
        "translated": "विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य है।",
        "target_language": "Hindi",
        "expected_label": 1,
        "case_type": "Admission ticket omitted",
        "required_facts": ["ID card requirement", "Admission ticket requirement"]
    },
    {
        "id": "C08",
        "original": "Students must bring their identity cards.",
        "translated": "மாணவர்கள் தங்கள் அடையாள அட்டைகளை கொண்டு வர வேண்டும்.",
        "target_language": "Tamil",
        "expected_label": 0,
        "case_type": "Alternative Tamil wording",
        "required_facts": ["ID card requirement"]
    },
    {
        "id": "C09",
        "original": "Students must bring their identity cards and admission tickets.",
        "translated": "மாணவர்கள் தங்கள் அடையாள அட்டைகளை கொண்டு வர வேண்டும்.",
        "target_language": "Tamil",
        "expected_label": 1,
        "case_type": "Admission ticket omitted",
        "required_facts": ["ID card requirement", "Admission ticket requirement"]
    },
    {
        "id": "C10",
        "original": "Do not share confidential information.",
        "translated": "தனிப்பட்ட தகவல்களை பகிர வேண்டாம்.",
        "target_language": "Tamil",
        "expected_label": 0,
        "case_type": "Negation preserved",
        "required_facts": []
    }
]

challenge_df = pd.DataFrame(challenge_data)

print("UNSEEN CHALLENGE SET CREATED!")
print("Total examples:", len(challenge_df))
display(challenge_df)


UNSEEN CHALLENGE SET CREATED!
Total examples: 10


,id,original,translated,target_language,expected_label,case_type,required_facts
0,C01,Bring three documents.,Bring 3 documents.,English,0,Written number equivalent,[]
1,C02,The event begins at 6 PM.,The event begins at 18:00.,English,0,Equivalent time format,[]
2,C03,Do not enter the room.,You are not allowed to enter the room.,English,0,Negation paraphrase,[]
3,C04,Bring three documents.,Bring four documents.,English,1,Written number changed,[]
4,C05,The meeting is at 6 PM.,The meeting is at 6 AM.,English,1,AM PM change,[]
5,C06,Students must bring their identity cards.,विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य...,Hindi,0,Alternative Hindi wording,[ID card requirement]
6,C07,Students must bring their identity cards and a...,विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य...,Hindi,1,Admission ticket omitted,"[ID card requirement, Admission ticket require..."
7,C08,Students must bring their identity cards.,மாணவர்கள் தங்கள் அடையாள அட்டைகளை கொண்டு வர வேண...,Tamil,0,Alternative Tamil wording,[ID card requirement]
8,C09,Students must bring their identity cards and a...,மாணவர்கள் தங்கள் அடையாள அட்டைகளை கொண்டு வர வேண...,Tamil,1,Admission ticket omitted,"[ID card requirement, Admission ticket require..."
9,C10,Do not share confidential information.,தனிப்பட்ட தகவல்களை பகிர வேண்டாம்.,Tamil,0,Negation preserved,[]


In [ ]:

challenge_results = []

for _, row in challenge_df.iterrows():

    result = meaningguard_analyze_v4(
        original=row["original"],
        translated=row["translated"],
        target_language=row["target_language"],
        required_facts=row["required_facts"]
    )

    predicted_label = (
        0 if result["decision"] == "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"
        else 1
    )

    challenge_results.append({
        "id": row["id"],
        "language": row["target_language"],
        "case_type": row["case_type"],
        "expected": row["expected_label"],
        "predicted": predicted_label,
        "decision": result["decision"],
        "correct": predicted_label == row["expected_label"]
    })

challenge_results_df = pd.DataFrame(challenge_results)

print("MEANINGGUARD — UNSEEN CHALLENGE RESULTS")
display(challenge_results_df)

print("\nCORRECT:", challenge_results_df["correct"].sum(),
      "/", len(challenge_results_df))

print("\nRESULTS BY LANGUAGE:")
display(
    challenge_results_df.groupby("language")["correct"]
    .agg(["sum", "count"])
)

print("\nFAILED CASES:")
display(
    challenge_results_df[
        challenge_results_df["correct"] == False
    ]
)


MEANINGGUARD — UNSEEN CHALLENGE RESULTS


,id,language,case_type,expected,predicted,decision,correct
0,C01,English,Written number equivalent,0,1,POTENTIAL MEANING CHANGE - VERIFY,False
1,C02,English,Equivalent time format,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
2,C03,English,Negation paraphrase,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
3,C04,English,Written number changed,1,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,False
4,C05,English,AM PM change,1,1,POTENTIAL MEANING CHANGE - VERIFY,True
5,C06,Hindi,Alternative Hindi wording,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
6,C07,Hindi,Admission ticket omitted,1,1,HUMAN REVIEW REQUIRED,True
7,C08,Tamil,Alternative Tamil wording,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
8,C09,Tamil,Admission ticket omitted,1,1,HUMAN REVIEW REQUIRED,True
9,C10,Tamil,Negation preserved,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True



CORRECT: 8 / 10

RESULTS BY LANGUAGE:


,sum,count
language,,
English,3,5
Hindi,2,2
Tamil,3,3



FAILED CASES:


,id,language,case_type,expected,predicted,decision,correct
0,C01,English,Written number equivalent,0,1,POTENTIAL MEANING CHANGE - VERIFY,False
3,C04,English,Written number changed,1,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,False


In [ ]:

import re

NUMBER_WORDS = {
    "zero": "0", "one": "1", "two": "2",
    "three": "3", "four": "4", "five": "5",
    "six": "6", "seven": "7", "eight": "8",
    "nine": "9", "ten": "10", "eleven": "11",
    "twelve": "12", "thirteen": "13", "fourteen": "14",
    "fifteen": "15", "sixteen": "16", "seventeen": "17",
    "eighteen": "18", "nineteen": "19", "twenty": "20"
}

def normalize_english_number_words(text):
    for word, digit in NUMBER_WORDS.items():
        text = re.sub(
            r"\b" + word + r"\b",
            digit,
            text,
            flags=re.IGNORECASE
        )
    return text

def meaningguard_analyze_v5(original, translated, target_language, required_facts):

    nli_result = analyze_bidirectionally(original, translated)

    normalized_original = normalize_english_number_words(original)
    normalized_translated = translated

    if target_language == "English":
        normalized_translated = normalize_english_number_words(translated)

    detail_result = detect_critical_changes_v2(
        normalized_original,
        normalized_translated,
        target_language
    )

    fact_results = None

    if target_language in LANGUAGE_FACT_PATTERNS:
        fact_results = check_language_facts(
            translated, required_facts, target_language
        )

    reasons = []

    time_status = detail_result["time_analysis"]["status"]
    number_changed = detail_result["number_analysis"]["number_changed"]
    negation_changed = detail_result["negation_analysis"]["negation_changed"]

    if time_status == "POSSIBLE TIME CHANGE":
        reasons.append("Possible time change detected")

    if number_changed:
        reasons.append("Possible numerical detail change detected")

    if negation_changed:
        reasons.append("Possible negation or instruction change")

    human_review_needed = False

    if (
        time_status == "TIME COULD NOT BE VERIFIED"
        and (
            contains_time_expression(original)
            or contains_time_expression(translated)
        )
    ):
        reasons.append("Time could not be verified")
        human_review_needed = True

    if fact_results is not None:
        for item in fact_results:
            if item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW":
                reasons.append("Expected phrase not found: " + item["fact"])
                human_review_needed = True

    possible_change = (
        time_status == "POSSIBLE TIME CHANGE"
        or number_changed
        or negation_changed
    )

    if possible_change:
        decision = "POTENTIAL MEANING CHANGE - VERIFY"
    elif human_review_needed:
        decision = "HUMAN REVIEW REQUIRED"
    else:
        decision = "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"

    return {
        "decision": decision,
        "reasons": reasons,
        "transformer_analysis": nli_result,
        "critical_detail_analysis": detail_result,
        "fact_verification": fact_results
    }

print("MeaningGuard V5 created successfully!")
print("Function available:", callable(meaningguard_analyze_v5))

MeaningGuard V5 created successfully!
Function available: True


In [ ]:
print(meaningguard_analyze_v5)

NameError: name 'meaningguard_analyze_v5' is not defined

In [ ]:

v5_results = []

for _, row in challenge_df.iterrows():

    result = meaningguard_analyze_v5(
        original=row["original"],
        translated=row["translated"],
        target_language=row["target_language"],
        required_facts=row["required_facts"]
    )

    predicted = (
        0 if result["decision"] == "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"
        else 1
    )

    v5_results.append({
        "id": row["id"],
        "case_type": row["case_type"],
        "expected": row["expected_label"],
        "predicted": predicted,
        "decision": result["decision"],
        "correct": predicted == row["expected_label"]
    })

v5_results_df = pd.DataFrame(v5_results)

display(v5_results_df)

print(
    "\nV5 CORRECT:",
    v5_results_df["correct"].sum(),
    "/",
    len(v5_results_df)
)


,id,case_type,expected,predicted,decision,correct
0,C01,Written number equivalent,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
1,C02,Equivalent time format,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
2,C03,Negation paraphrase,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
3,C04,Written number changed,1,1,POTENTIAL MEANING CHANGE - VERIFY,True
4,C05,AM PM change,1,1,POTENTIAL MEANING CHANGE - VERIFY,True
5,C06,Alternative Hindi wording,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
6,C07,Admission ticket omitted,1,1,HUMAN REVIEW REQUIRED,True
7,C08,Alternative Tamil wording,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
8,C09,Admission ticket omitted,1,1,HUMAN REVIEW REQUIRED,True
9,C10,Negation preserved,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True



V5 CORRECT: 10 / 10


In [ ]:

import zipfile

v5_results_df.to_csv(
    "/content/meaningguard_v5_challenge_results.csv",
    index=False
)

with zipfile.ZipFile(
    "/content/MeaningGuard_V5_Results.zip", "w"
) as z:
    z.write("/content/meaningguard_v5_challenge_results.csv")

print("V5 results saved successfully!")
from google.colab import files
files.download("/content/MeaningGuard_V5_Results.zip")


V5 results saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

independent_data = [
    {
        "id": "I01",
        "original": "Thirty students must attend the workshop.",
        "translated": "30 students must attend the workshop.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Unseen number equivalent",
        "required_facts": []
    },
    {
        "id": "I02",
        "original": "Thirty students must attend the workshop.",
        "translated": "40 students must attend the workshop.",
        "target_language": "English",
        "expected_label": 1,
        "case_type": "Unseen number changed",
        "required_facts": []
    },
    {
        "id": "I03",
        "original": "Submit at least 5 assignments.",
        "translated": "Submit more than 5 assignments.",
        "target_language": "English",
        "expected_label": 1,
        "case_type": "Subtle quantity change",
        "required_facts": []
    },
    {
        "id": "I04",
        "original": "You must not share your password.",
        "translated": "You should refrain from sharing your password.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Alternative negation wording",
        "required_facts": []
    },
    {
        "id": "I05",
        "original": "The examination begins at 9:30 AM.",
        "translated": "The examination begins at 09:30.",
        "target_language": "English",
        "expected_label": 0,
        "case_type": "Equivalent time format",
        "required_facts": []
    },
    {
        "id": "I06",
        "original": "Do not share your password.",
        "translated": "कृपया अपना पासवर्ड साझा करने से बचें।",
        "target_language": "Hindi",
        "expected_label": 0,
        "case_type": "Hindi negation paraphrase",
        "required_facts": []
    },
    {
        "id": "I07",
        "original": "The meeting starts at 4 PM.",
        "translated": "बैठक शाम 5 बजे शुरू होगी।",
        "target_language": "Hindi",
        "expected_label": 1,
        "case_type": "Hindi time changed",
        "required_facts": []
    },
    {
        "id": "I08",
        "original": "Students must bring their ID cards and certificates.",
        "translated": "विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य है।",
        "target_language": "Hindi",
        "expected_label": 1,
        "case_type": "Hindi information omitted",
        "required_facts": []
    },
    {
        "id": "I09",
        "original": "Do not share confidential information.",
        "translated": "ரகசியமான தகவல்களைப் பகிர்வதைத் தவிர்க்கவும்.",
        "target_language": "Tamil",
        "expected_label": 0,
        "case_type": "Tamil negation paraphrase",
        "required_facts": []
    },
    {
        "id": "I10",
        "original": "Students must bring their ID cards and certificates.",
        "translated": "மாணவர்கள் தங்கள் அடையாள அட்டைகளை கொண்டு வர வேண்டும்.",
        "target_language": "Tamil",
        "expected_label": 1,
        "case_type": "Tamil information omitted",
        "required_facts": []
    }
]

independent_df = pd.DataFrame(independent_data)

print("INDEPENDENT EVALUATION SET CREATED")
print("Total examples:", len(independent_df))
print("\nCases by language:")
print(independent_df["target_language"].value_counts())

display(independent_df)


INDEPENDENT EVALUATION SET CREATED
Total examples: 10

Cases by language:
target_language
English    5
Hindi      3
Tamil      2
Name: count, dtype: int64


,id,original,translated,target_language,expected_label,case_type,required_facts
0,I01,Thirty students must attend the workshop.,30 students must attend the workshop.,English,0,Unseen number equivalent,[]
1,I02,Thirty students must attend the workshop.,40 students must attend the workshop.,English,1,Unseen number changed,[]
2,I03,Submit at least 5 assignments.,Submit more than 5 assignments.,English,1,Subtle quantity change,[]
3,I04,You must not share your password.,You should refrain from sharing your password.,English,0,Alternative negation wording,[]
4,I05,The examination begins at 9:30 AM.,The examination begins at 09:30.,English,0,Equivalent time format,[]
5,I06,Do not share your password.,कृपया अपना पासवर्ड साझा करने से बचें।,Hindi,0,Hindi negation paraphrase,[]
6,I07,The meeting starts at 4 PM.,बैठक शाम 5 बजे शुरू होगी।,Hindi,1,Hindi time changed,[]
7,I08,Students must bring their ID cards and certifi...,विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य...,Hindi,1,Hindi information omitted,[]
8,I09,Do not share confidential information.,ரகசியமான தகவல்களைப் பகிர்வதைத் தவிர்க்கவும்.,Tamil,0,Tamil negation paraphrase,[]
9,I10,Students must bring their ID cards and certifi...,மாணவர்கள் தங்கள் அடையாள அட்டைகளை கொண்டு வர வேண...,Tamil,1,Tamil information omitted,[]


In [ ]:

independent_results = []

for _, row in independent_df.iterrows():

    result = meaningguard_analyze_v5(
        original=row["original"],
        translated=row["translated"],
        target_language=row["target_language"],
        required_facts=row["required_facts"]
    )

    predicted = (
        0 if result["decision"] == "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"
        else 1
    )

    independent_results.append({
        "id": row["id"],
        "language": row["target_language"],
        "case_type": row["case_type"],
        "expected": row["expected_label"],
        "predicted": predicted,
        "decision": result["decision"],
        "correct": predicted == row["expected_label"]
    })

independent_results_df = pd.DataFrame(independent_results)

print("MEANINGGUARD V5 — INDEPENDENT EVALUATION")
display(independent_results_df)

print("\nCORRECT PREDICTIONS:",
      independent_results_df["correct"].sum(),
      "/", len(independent_results_df))

print("\nRESULTS BY LANGUAGE:")
display(
    independent_results_df.groupby("language")["correct"]
    .agg(["sum", "count"])
)

print("\nFAILED CASES:")
display(
    independent_results_df[
        independent_results_df["correct"] == False
    ]
)


MEANINGGUARD V5 — INDEPENDENT EVALUATION


,id,language,case_type,expected,predicted,decision,correct
0,I01,English,Unseen number equivalent,0,1,POTENTIAL MEANING CHANGE - VERIFY,False
1,I02,English,Unseen number changed,1,1,POTENTIAL MEANING CHANGE - VERIFY,True
2,I03,English,Subtle quantity change,1,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,False
3,I04,English,Alternative negation wording,0,1,POTENTIAL MEANING CHANGE - VERIFY,False
4,I05,English,Equivalent time format,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,True
5,I06,Hindi,Hindi negation paraphrase,0,1,POTENTIAL MEANING CHANGE - VERIFY,False
6,I07,Hindi,Hindi time changed,1,1,POTENTIAL MEANING CHANGE - VERIFY,True
7,I08,Hindi,Hindi information omitted,1,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,False
8,I09,Tamil,Tamil negation paraphrase,0,1,POTENTIAL MEANING CHANGE - VERIFY,False
9,I10,Tamil,Tamil information omitted,1,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,False



CORRECT PREDICTIONS: 3 / 10

RESULTS BY LANGUAGE:


,sum,count
language,,
English,2,5
Hindi,1,3
Tamil,0,2



FAILED CASES:


,id,language,case_type,expected,predicted,decision,correct
0,I01,English,Unseen number equivalent,0,1,POTENTIAL MEANING CHANGE - VERIFY,False
2,I03,English,Subtle quantity change,1,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,False
3,I04,English,Alternative negation wording,0,1,POTENTIAL MEANING CHANGE - VERIFY,False
5,I06,Hindi,Hindi negation paraphrase,0,1,POTENTIAL MEANING CHANGE - VERIFY,False
7,I08,Hindi,Hindi information omitted,1,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,False
8,I09,Tamil,Tamil negation paraphrase,0,1,POTENTIAL MEANING CHANGE - VERIFY,False
9,I10,Tamil,Tamil information omitted,1,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,False


In [ ]:

diagnostic_results = []

for _, row in independent_df.iterrows():

    result = meaningguard_analyze_v5(
        original=row["original"],
        translated=row["translated"],
        target_language=row["target_language"],
        required_facts=row["required_facts"]
    )

    detail = result["critical_detail_analysis"]

    diagnostic_results.append({
        "id": row["id"],
        "decision": result["decision"],
        "reasons": "; ".join(result["reasons"]),
        "time_check": detail["time_analysis"]["status"],
        "number_changed": detail["number_analysis"]["number_changed"],
        "negation_changed": detail["negation_analysis"]["negation_changed"],
        "forward_NLI": result["transformer_analysis"]["forward"]["prediction"],
        "backward_NLI": result["transformer_analysis"]["backward"]["prediction"]
    })

diagnostic_df = pd.DataFrame(diagnostic_results)

display(diagnostic_df)


,id,decision,reasons,time_check,number_changed,negation_changed,forward_NLI,backward_NLI
0,I01,POTENTIAL MEANING CHANGE - VERIFY,Possible numerical detail change detected,TIME COULD NOT BE VERIFIED,True,False,entailment,entailment
1,I02,POTENTIAL MEANING CHANGE - VERIFY,Possible numerical detail change detected,TIME COULD NOT BE VERIFIED,True,False,contradiction,contradiction
2,I03,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,,TIME COULD NOT BE VERIFIED,False,False,contradiction,entailment
3,I04,POTENTIAL MEANING CHANGE - VERIFY,Possible negation or instruction change,TIME COULD NOT BE VERIFIED,False,True,entailment,entailment
4,I05,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,,TIME PRESERVED,False,False,entailment,neutral
5,I06,POTENTIAL MEANING CHANGE - VERIFY,Possible negation or instruction change,TIME COULD NOT BE VERIFIED,False,True,entailment,entailment
6,I07,POTENTIAL MEANING CHANGE - VERIFY,Possible time change detected,POSSIBLE TIME CHANGE,False,False,entailment,contradiction
7,I08,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,,TIME COULD NOT BE VERIFIED,False,False,entailment,neutral
8,I09,POTENTIAL MEANING CHANGE - VERIFY,Possible negation or instruction change,TIME COULD NOT BE VERIFIED,False,True,entailment,entailment
9,I10,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,,TIME COULD NOT BE VERIFIED,False,False,entailment,entailment


In [ ]:

nli_probability_results = []

for _, row in independent_df.iterrows():

    result = analyze_bidirectionally(
        row["original"],
        row["translated"]
    )

    forward = result["forward"]["probabilities"]
    backward = result["backward"]["probabilities"]

    nli_probability_results.append({
        "id": row["id"],
        "language": row["target_language"],
        "forward_entailment": round(forward["entailment"], 4),
        "backward_entailment": round(backward["entailment"], 4),
        "forward_neutral": round(forward["neutral"], 4),
        "backward_neutral": round(backward["neutral"], 4),
        "forward_contradiction": round(forward["contradiction"], 4),
        "backward_contradiction": round(backward["contradiction"], 4)
    })

nli_probability_df = pd.DataFrame(nli_probability_results)

display(nli_probability_df)

print("NLI probability inspection completed.")


,id,language,forward_entailment,backward_entailment,forward_neutral,backward_neutral,forward_contradiction,backward_contradiction
0,I01,English,0.9980,0.9976,0.0019,0.0020,0.0002,0.0003
1,I02,English,0.0333,0.0282,0.0115,0.0194,0.9551,0.9526
2,I03,English,0.2979,0.9585,0.3008,0.0272,0.4014,0.0142
3,I04,English,0.9946,0.9971,0.0050,0.0026,0.0002,0.0002
4,I05,English,0.9946,0.1884,0.0048,0.8105,0.0006,0.0011
5,I06,Hindi,0.9990,0.9985,0.0011,0.0011,0.0001,0.0001
6,I07,Hindi,0.5713,0.0401,0.0157,0.0119,0.4131,0.9478
7,I08,Hindi,0.9990,0.2050,0.0010,0.7935,0.0001,0.0016
8,I09,Tamil,0.9990,0.9985,0.0009,0.0013,0.0001,0.0001
9,I10,Tamil,0.9990,0.9775,0.0007,0.0221,0.0002,0.0003


NLI probability inspection completed.


In [ ]:

def interpret_nli_signals(result):

    forward = result["forward"]["probabilities"]
    backward = result["backward"]["probabilities"]

    f_ent = forward["entailment"]
    b_ent = backward["entailment"]

    f_con = forward["contradiction"]
    b_con = backward["contradiction"]

    f_neu = forward["neutral"]
    b_neu = backward["neutral"]

    reasons = []

    # Strong contradiction evidence
    if max(f_con, b_con) >= 0.80:
        reasons.append("Strong contradiction signal")

    # Possible directional omission signal
    if f_ent >= 0.75 and b_neu >= 0.50:
        reasons.append("Possible information omission signal")

    # Conflicting directional evidence
    if abs(f_ent - b_ent) >= 0.50:
        reasons.append("Strong directional asymmetry")

    # Conflicting evidence should not be treated as proof
    if not reasons:
        status = "NO STRONG NLI WARNING"
    else:
        status = "NLI HUMAN REVIEW CANDIDATE"

    return {
        "status": status,
        "reasons": reasons,
        "forward_entailment": round(f_ent, 4),
        "backward_entailment": round(b_ent, 4)
    }

nli_diagnostic_results = []

for _, row in independent_df.iterrows():

    result = analyze_bidirectionally(
        row["original"],
        row["translated"]
    )

    interpreted = interpret_nli_signals(result)

    nli_diagnostic_results.append({
        "id": row["id"],
        "language": row["target_language"],
        "status": interpreted["status"],
        "reasons": "; ".join(interpreted["reasons"])
    })

nli_diagnostic_df = pd.DataFrame(nli_diagnostic_results)

display(nli_diagnostic_df)

print("NLI SIGNAL ANALYSIS COMPLETE")

,id,language,status,reasons
0,I01,English,NO STRONG NLI WARNING,
1,I02,English,NLI HUMAN REVIEW CANDIDATE,Strong contradiction signal
2,I03,English,NLI HUMAN REVIEW CANDIDATE,Strong directional asymmetry
3,I04,English,NO STRONG NLI WARNING,
4,I05,English,NLI HUMAN REVIEW CANDIDATE,Possible information omission signal; Strong d...
5,I06,Hindi,NO STRONG NLI WARNING,
6,I07,Hindi,NLI HUMAN REVIEW CANDIDATE,Strong contradiction signal; Strong directiona...
7,I08,Hindi,NLI HUMAN REVIEW CANDIDATE,Possible information omission signal; Strong d...
8,I09,Tamil,NO STRONG NLI WARNING,
9,I10,Tamil,NO STRONG NLI WARNING,


NLI SIGNAL ANALYSIS COMPLETE


In [ ]:

# MEANINGGUARD V6 — EVIDENCE-BASED DECISION ENGINE

import re

# Extend the existing English number dictionary
NUMBER_WORDS["thirty"] = "30"

def meaningguard_analyze_v6(
    original, translated, target_language, required_facts
):

    # Layer 1: Normalise known English number words
    normalized_original = normalize_english_number_words(original)
    normalized_translated = translated

    if target_language == "English":
        normalized_translated = normalize_english_number_words(
            translated
        )

    # Layer 2: Critical detail checks
    detail = detect_critical_changes_v2(
        normalized_original,
        normalized_translated,
        target_language
    )

    # Layer 3: Transformer analysis
    nli = analyze_bidirectionally(original, translated)

    forward = nli["forward"]["probabilities"]
    backward = nli["backward"]["probabilities"]

    # Layer 4: Optional phrase-based fact verification
    fact_results = None

    if target_language in LANGUAGE_FACT_PATTERNS:
        fact_results = check_language_facts(
            translated, required_facts, target_language
        )

    reasons = []

    time_status = detail["time_analysis"]["status"]
    number_changed = detail["number_analysis"]["number_changed"]
    negation_changed = detail["negation_analysis"]["negation_changed"]

    # NLI signals
    strong_contradiction = (
        max(
            forward["contradiction"],
            backward["contradiction"]
        ) >= 0.80
    )

    possible_asymmetry = (
        abs(
            forward["entailment"] -
            backward["entailment"]
        ) >= 0.50
    )

    possible_omission = (
        forward["entailment"] >= 0.75
        and backward["neutral"] >= 0.50
    )

    # Avoid treating known equivalent time formats
    # as omission signals.
    suppress_time_asymmetry = (
        time_status == "TIME PRESERVED"
    )

    if suppress_time_asymmetry:
        possible_asymmetry = False
        possible_omission = False

    # Evidence collection
    if time_status == "POSSIBLE TIME CHANGE":
        reasons.append("Possible time change")

    if number_changed:
        reasons.append("Numerical details differ")

    if negation_changed:
        reasons.append("Possible instruction or negation difference")

    if strong_contradiction:
        reasons.append("Strong transformer contradiction signal")

    if possible_omission:
        reasons.append("Possible information omission signal")

    elif possible_asymmetry:
        reasons.append("Unusual directional NLI asymmetry")

    if fact_results is not None:
        for item in fact_results:
            if item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW":
                reasons.append(
                    "Expected phrase requires review: " + item["fact"]
                )

    # Decision policy
    human_review_needed = (
        number_changed
        or negation_changed
        or possible_asymmetry
        or possible_omission
        or (
            time_status == "TIME COULD NOT BE VERIFIED"
            and (
                contains_time_expression(original)
                or contains_time_expression(translated)
            )
        )
        or (
            fact_results is not None
            and any(
                item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW"
                for item in fact_results
            )
        )
    )

    if strong_contradiction:
        decision = "POTENTIAL MEANING CHANGE - VERIFY"

    elif human_review_needed:
        decision = "HUMAN REVIEW REQUIRED"

    else:
        decision = "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"

    return {
        "decision": decision,
        "reasons": reasons,
        "critical_detail_analysis": detail,
        "transformer_analysis": nli,
        "fact_verification": fact_results
    }


print("MeaningGuard V6 created successfully!")


MeaningGuard V6 created successfully!


In [ ]:

# MEANINGGUARD V6 — EVIDENCE-BASED DECISION ENGINE

import re

# Extend the existing English number dictionary
NUMBER_WORDS["thirty"] = "30"

def meaningguard_analyze_v6(
    original, translated, target_language, required_facts
):

    # Layer 1: Normalise known English number words
    normalized_original = normalize_english_number_words(original)
    normalized_translated = translated

    if target_language == "English":
        normalized_translated = normalize_english_number_words(
            translated
        )

    # Layer 2: Critical detail checks
    detail = detect_critical_changes_v2(
        normalized_original,
        normalized_translated,
        target_language
    )

    # Layer 3: Transformer analysis
    nli = analyze_bidirectionally(original, translated)

    forward = nli["forward"]["probabilities"]
    backward = nli["backward"]["probabilities"]

    # Layer 4: Optional phrase-based fact verification
    fact_results = None

    if target_language in LANGUAGE_FACT_PATTERNS:
        fact_results = check_language_facts(
            translated, required_facts, target_language
        )

    reasons = []

    time_status = detail["time_analysis"]["status"]
    number_changed = detail["number_analysis"]["number_changed"]
    negation_changed = detail["negation_analysis"]["negation_changed"]

    # NLI signals
    strong_contradiction = (
        max(
            forward["contradiction"],
            backward["contradiction"]
        ) >= 0.80
    )

    possible_asymmetry = (
        abs(
            forward["entailment"] -
            backward["entailment"]
        ) >= 0.50
    )

    possible_omission = (
        forward["entailment"] >= 0.75
        and backward["neutral"] >= 0.50
    )

    # Avoid treating known equivalent time formats
    # as omission signals.
    suppress_time_asymmetry = (
        time_status == "TIME PRESERVED"
    )

    if suppress_time_asymmetry:
        possible_asymmetry = False
        possible_omission = False

    # Evidence collection
    if time_status == "POSSIBLE TIME CHANGE":
        reasons.append("Possible time change")

    if number_changed:
        reasons.append("Numerical details differ")

    if negation_changed:
        reasons.append("Possible instruction or negation difference")

    if strong_contradiction:
        reasons.append("Strong transformer contradiction signal")

    if possible_omission:
        reasons.append("Possible information omission signal")

    elif possible_asymmetry:
        reasons.append("Unusual directional NLI asymmetry")

    if fact_results is not None:
        for item in fact_results:
            if item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW":
                reasons.append(
                    "Expected phrase requires review: " + item["fact"]
                )

    # Decision policy
    human_review_needed = (
        number_changed
        or negation_changed
        or possible_asymmetry
        or possible_omission
        or (
            time_status == "TIME COULD NOT BE VERIFIED"
            and (
                contains_time_expression(original)
                or contains_time_expression(translated)
            )
        )
        or (
            fact_results is not None
            and any(
                item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW"
                for item in fact_results
            )
        )
    )

    if strong_contradiction:
        decision = "POTENTIAL MEANING CHANGE - VERIFY"

    elif human_review_needed:
        decision = "HUMAN REVIEW REQUIRED"

    else:
        decision = "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"

    return {
        "decision": decision,
        "reasons": reasons,
        "critical_detail_analysis": detail,
        "transformer_analysis": nli,
        "fact_verification": fact_results
    }


print("MeaningGuard V6 created successfully!")


MeaningGuard V6 created successfully!


In [ ]:

v6_development_results = []

for _, row in independent_df.iterrows():

    result = meaningguard_analyze_v6(
        original=row["original"],
        translated=row["translated"],
        target_language=row["target_language"],
        required_facts=row["required_facts"]
    )

    v6_development_results.append({
        "id": row["id"],
        "expected": row["expected_label"],
        "decision": result["decision"],
        "reasons": "; ".join(result["reasons"])
    })

v6_development_df = pd.DataFrame(v6_development_results)

display(v6_development_df)

print("\nV6 DEVELOPMENT CHECK COMPLETED")


,id,expected,decision,reasons
0,I01,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,
1,I02,1,POTENTIAL MEANING CHANGE - VERIFY,Numerical details differ; Strong transformer c...
2,I03,1,HUMAN REVIEW REQUIRED,Unusual directional NLI asymmetry
3,I04,0,HUMAN REVIEW REQUIRED,Possible instruction or negation difference
4,I05,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,
5,I06,0,HUMAN REVIEW REQUIRED,Possible instruction or negation difference
6,I07,1,POTENTIAL MEANING CHANGE - VERIFY,Possible time change; Strong transformer contr...
7,I08,1,HUMAN REVIEW REQUIRED,Possible information omission signal
8,I09,0,HUMAN REVIEW REQUIRED,Possible instruction or negation difference
9,I10,1,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,



V6 DEVELOPMENT CHECK COMPLETED


In [ ]:

# MEANINGGUARD — SOURCE-GROUNDED FACT CHECKING

# Extend the existing multilingual fact dictionary
LANGUAGE_FACT_PATTERNS["Hindi"]["Certificate requirement"] = [
    "प्रमाणपत्र",
    "प्रमाण पत्र",
    "सर्टिफिकेट"
]

LANGUAGE_FACT_PATTERNS["Tamil"]["Certificate requirement"] = [
    "சான்றிதழ்",
    "சான்றிதழ்களை",
    "சான்றிதழ்களையும்"
]


def derive_required_facts(original):

    source = original.casefold()
    facts = []

    # Identify requirements explicitly mentioned in English
    if re.search(r"\b(id cards?|identity cards?)\b", source):
        facts.append("ID card requirement")

    if re.search(r"\bcertificates?\b", source):
        facts.append("Certificate requirement")

    if re.search(r"\badmission tickets?\b", source):
        facts.append("Admission ticket requirement")

    return facts


def verify_source_facts(original, translated, target_language):

    required_facts = derive_required_facts(original)

    if target_language not in LANGUAGE_FACT_PATTERNS:
        return {
            "required_facts": required_facts,
            "verification": "HUMAN REVIEW REQUIRED",
            "details": []
        }

    results = check_language_facts(
        translated,
        required_facts,
        target_language
    )

    missing = [
        item["fact"]
        for item in results
        if item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW"
    ]

    return {
        "required_facts": required_facts,
        "verification": (
            "HUMAN REVIEW REQUIRED" if missing
            else "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"
        ),
        "details": results
    }


# Test the previously missed Hindi and Tamil examples
for case_id in ["I08", "I10"]:

    row = independent_df[
        independent_df["id"] == case_id
    ].iloc[0]

    result = verify_source_facts(
        row["original"],
        row["translated"],
        row["target_language"]
    )

    print("\nCASE:", case_id)
    print("Required facts:", result["required_facts"])
    print("Verification:", result["verification"])
    print("Details:", result["details"])



CASE: I08
Required facts: ['ID card requirement', 'Certificate requirement']
Verification: HUMAN REVIEW REQUIRED
Details: [{'fact': 'ID card requirement', 'status': 'PHRASE FOUND'}, {'fact': 'Certificate requirement', 'status': 'PHRASE NOT FOUND - HUMAN REVIEW'}]

CASE: I10
Required facts: ['ID card requirement', 'Certificate requirement']
Verification: HUMAN REVIEW REQUIRED
Details: [{'fact': 'ID card requirement', 'status': 'PHRASE FOUND'}, {'fact': 'Certificate requirement', 'status': 'PHRASE NOT FOUND - HUMAN REVIEW'}]


# 7. MeaningGuard Hybrid Decision Engine — V7

In [ ]:

v7_development_results = []

for _, row in independent_df.iterrows():

    result = meaningguard_analyze_v7(
        original=row["original"],
        translated=row["translated"],
        target_language=row["target_language"],
        required_facts=row["required_facts"]
    )

    v7_development_results.append({
        "id": row["id"],
        "expected": row["expected_label"],
        "decision": result["decision"],
        "source_facts": ", ".join(result["derived_source_facts"]),
        "reasons": "; ".join(result["reasons"])
    })

v7_development_df = pd.DataFrame(v7_development_results)

display(v7_development_df)

print("\nV7 DEVELOPMENT TEST COMPLETED")


NameError: name 'meaningguard_analyze_v7' is not defined

In [ ]:
print("V7 exists:", "meaningguard_analyze_v7" in globals())

print("Function object:", globals().get("meaningguard_analyze_v7"))

V7 exists: False
Function object: None


In [ ]:

# MEANINGGUARD V7 — FUNCTION RECOVERY

def meaningguard_analyze_v7(
    original, translated, target_language, required_facts
):
    normalized_original = normalize_english_number_words(original)
    normalized_translated = translated

    if target_language == "English":
        normalized_translated = normalize_english_number_words(translated)

    detail = detect_critical_changes_v2(
        normalized_original,
        normalized_translated,
        target_language
    )

    nli = analyze_bidirectionally(original, translated)
    forward = nli["forward"]["probabilities"]
    backward = nli["backward"]["probabilities"]

    derived_facts = derive_required_facts(original)
    all_facts = list(dict.fromkeys(
        (required_facts or []) + derived_facts
    ))

    fact_results = None

    if target_language in LANGUAGE_FACT_PATTERNS:
        fact_results = check_language_facts(
            translated, all_facts, target_language
        )

    time_status = detail["time_analysis"]["status"]
    number_changed = detail["number_analysis"]["number_changed"]
    negation_changed = detail["negation_analysis"]["negation_changed"]

    strong_contradiction = max(
        forward["contradiction"],
        backward["contradiction"]
    ) >= 0.80

    possible_asymmetry = abs(
        forward["entailment"] - backward["entailment"]
    ) >= 0.50

    possible_omission = (
        forward["entailment"] >= 0.75
        and backward["neutral"] >= 0.50
    )

    if time_status == "TIME PRESERVED":
        possible_asymmetry = False
        possible_omission = False

    reasons = []

    if time_status == "POSSIBLE TIME CHANGE":
        reasons.append("Possible time change")
    if number_changed:
        reasons.append("Numerical details differ")
    if negation_changed:
        reasons.append("Possible instruction or negation difference")
    if strong_contradiction:
        reasons.append("Strong transformer contradiction signal")
    if possible_omission:
        reasons.append("Possible information omission signal")
    elif possible_asymmetry:
        reasons.append("Unusual directional NLI asymmetry")

    missing_facts = []

    if fact_results is not None:
        missing_facts = [
            item["fact"] for item in fact_results
            if item["status"] == "PHRASE NOT FOUND - HUMAN REVIEW"
        ]

        if missing_facts:
            reasons.append(
                "Expected source facts require review: "
                + ", ".join(missing_facts)
            )

    time_uncertain = (
        time_status == "TIME COULD NOT BE VERIFIED"
        and (
            contains_time_expression(original)
            or contains_time_expression(translated)
        )
    )

    human_review_needed = (
        number_changed or negation_changed
        or possible_asymmetry or possible_omission
        or time_uncertain or bool(missing_facts)
    )

    if strong_contradiction or time_status == "POSSIBLE TIME CHANGE":
        decision = "POTENTIAL MEANING CHANGE - VERIFY"
    elif human_review_needed:
        decision = "HUMAN REVIEW REQUIRED"
    else:
        decision = "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"

    return {
        "decision": decision,
        "reasons": reasons,
        "derived_source_facts": derived_facts,
        "critical_detail_analysis": detail,
        "transformer_analysis": nli,
        "fact_verification": fact_results
    }


print("V7 exists:", "meaningguard_analyze_v7" in globals())
print("MeaningGuard V7 recovery successful!")


V7 exists: True
MeaningGuard V7 recovery successful!


In [ ]:

v7_development_results = []

for _, row in independent_df.iterrows():

    result = meaningguard_analyze_v7(
        original=row["original"],
        translated=row["translated"],
        target_language=row["target_language"],
        required_facts=row["required_facts"]
    )

    v7_development_results.append({
        "id": row["id"],
        "expected": row["expected_label"],
        "decision": result["decision"],
        "source_facts": ", ".join(result["derived_source_facts"]),
        "reasons": "; ".join(result["reasons"])
    })

v7_development_df = pd.DataFrame(v7_development_results)

display(v7_development_df)

print("\nV7 DEVELOPMENT TEST COMPLETED")


,id,expected,decision,source_facts,reasons
0,I01,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,,
1,I02,1,POTENTIAL MEANING CHANGE - VERIFY,,Numerical details differ; Strong transformer c...
2,I03,1,HUMAN REVIEW REQUIRED,,Unusual directional NLI asymmetry
3,I04,0,HUMAN REVIEW REQUIRED,,Possible instruction or negation difference
4,I05,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,,
5,I06,0,HUMAN REVIEW REQUIRED,,Possible instruction or negation difference
6,I07,1,POTENTIAL MEANING CHANGE - VERIFY,,Possible time change; Strong transformer contr...
7,I08,1,HUMAN REVIEW REQUIRED,"ID card requirement, Certificate requirement",Possible information omission signal; Expected...
8,I09,0,HUMAN REVIEW REQUIRED,,Possible instruction or negation difference
9,I10,1,HUMAN REVIEW REQUIRED,"ID card requirement, Certificate requirement",Expected source facts require review: Certific...



V7 DEVELOPMENT TEST COMPLETED


# 8. Preliminary Holdout Evaluation Dataset

In [74]:

# MEANINGGUARD — FRESH HOLDOUT DATASET
# Keep this dataset separate. Do not tune V7 using these cases.

import pandas as pd

holdout_data = [
    # ENGLISH — 10 cases
    ["E01", "The office opens at 9 AM.", "The office opens at 09:00.", "English", 0, "Equivalent time"],
    ["E02", "Submit 4 forms.", "Submit 5 forms.", "English", 1, "Changed number"],
    ["E03", "Do not enter the room.", "Entry is not permitted in the room.", "English", 0, "Negation paraphrase"],
    ["E04", "Do not share your password.", "Share your password.", "English", 1, "Negation changed"],
    ["E05", "Submit at least 6 assignments.", "Submit more than 6 assignments.", "English", 1, "Quantity meaning changed"],
    ["E06", "Bring your ID and signed form.", "Bring the signed form and your ID.", "English", 0, "Reordered information"],
    ["E07", "Bring your ID and signed form.", "Bring your ID.", "English", 1, "Information omitted"],
    ["E08", "Bring three copies.", "Bring 3 copies.", "English", 0, "Number format"],
    ["E09", "The library closes at 6 PM.", "The library closes at 6 AM.", "English", 1, "Changed time"],
    ["E10", "The application must be submitted by Friday.", "Friday is the deadline for the application.", "English", 0, "Meaning-preserving paraphrase"],

    # HINDI — 10 cases
    ["H01", "Students must bring their identity card and certificate.", "विद्यार्थियों को अपना पहचान पत्र और प्रमाणपत्र लाना अनिवार्य है।", "Hindi", 0, "Complete requirements"],
    ["H02", "Students must bring their identity card and certificate.", "विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य है।", "Hindi", 1, "Certificate omitted"],
    ["H03", "The meeting starts at 4 PM.", "बैठक शाम 4 बजे शुरू होगी।", "Hindi", 0, "Time preserved"],
    ["H04", "The meeting starts at 4 PM.", "बैठक शाम 5 बजे शुरू होगी।", "Hindi", 1, "Time changed"],
    ["H05", "Bring 3 copies of the document.", "दस्तावेज़ की 3 प्रतियां लाएँ।", "Hindi", 0, "Number preserved"],
    ["H06", "Bring 3 copies of the document.", "दस्तावेज़ की 4 प्रतियां लाएँ।", "Hindi", 1, "Number changed"],
    ["H07", "Do not share your password.", "कृपया अपना पासवर्ड साझा न करें।", "Hindi", 0, "Negation paraphrase"],
    ["H08", "Do not share your password.", "कृपया अपना पासवर्ड साझा करें।", "Hindi", 1, "Negation changed"],
    ["H09", "Students must bring their identity card.", "विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य है।", "Hindi", 0, "Requirement preserved"],
    ["H10", "Students must bring their identity card.", "विद्यार्थियों को अपना प्रवेश पत्र लाना अनिवार्य है।", "Hindi", 1, "Requirement changed"],

    # TAMIL — 10 cases
    ["T01", "Students must bring their ID card and certificate.", "மாணவர்கள் தங்கள் அடையாள அட்டையையும் சான்றிதழையும் கொண்டு வர வேண்டும்.", "Tamil", 0, "Complete requirements"],
    ["T02", "Students must bring their ID card and certificate.", "மாணவர்கள் தங்கள் அடையாள அட்டையை கொண்டு வர வேண்டும்.", "Tamil", 1, "Certificate omitted"],
    ["T03", "The meeting starts at 4 PM.", "கூட்டம் மாலை 4 மணிக்கு தொடங்கும்.", "Tamil", 0, "Time preserved"],
    ["T04", "The meeting starts at 4 PM.", "கூட்டம் மாலை 5 மணிக்கு தொடங்கும்.", "Tamil", 1, "Time changed"],
    ["T05", "Bring 3 copies of the document.", "ஆவணத்தின் 3 பிரதிகளைக் கொண்டு வாருங்கள்.", "Tamil", 0, "Number preserved"],
    ["T06", "Bring 3 copies of the document.", "ஆவணத்தின் 4 பிரதிகளைக் கொண்டு வாருங்கள்.", "Tamil", 1, "Number changed"],
    ["T07", "Do not share confidential information.", "ரகசியமான தகவல்களைப் பகிர வேண்டாம்.", "Tamil", 0, "Negation preserved"],
    ["T08", "Do not share confidential information.", "ரகசியமான தகவல்களைப் பகிரவும்.", "Tamil", 1, "Negation changed"],
    ["T09", "Students must bring their ID cards.", "மாணவர்கள் தங்கள் அடையாள அட்டைகளை கொண்டு வர வேண்டும்.", "Tamil", 0, "Requirement preserved"],
    ["T10", "Students must bring their ID cards.", "மாணவர்கள் தங்கள் சான்றிதழ்களை கொண்டு வர வேண்டும்.", "Tamil", 1, "Requirement changed"]
]

holdout_df = pd.DataFrame(
    holdout_data,
    columns=[
        "id", "original", "translated",
        "target_language", "expected_label", "case_type"
    ]
)

holdout_df["required_facts"] = [[] for _ in range(len(holdout_df))]

print("Fresh holdout dataset created!")
print("Total examples:", len(holdout_df))
print("\nLanguage distribution:")
print(holdout_df["target_language"].value_counts())

display(holdout_df)


Fresh holdout dataset created!
Total examples: 30

Language distribution:
target_language
English    10
Hindi      10
Tamil      10
Name: count, dtype: int64


,id,original,translated,target_language,expected_label,case_type,required_facts
0,E01,The office opens at 9 AM.,The office opens at 09:00.,English,0,Equivalent time,[]
1,E02,Submit 4 forms.,Submit 5 forms.,English,1,Changed number,[]
2,E03,Do not enter the room.,Entry is not permitted in the room.,English,0,Negation paraphrase,[]
3,E04,Do not share your password.,Share your password.,English,1,Negation changed,[]
4,E05,Submit at least 6 assignments.,Submit more than 6 assignments.,English,1,Quantity meaning changed,[]
5,E06,Bring your ID and signed form.,Bring the signed form and your ID.,English,0,Reordered information,[]
6,E07,Bring your ID and signed form.,Bring your ID.,English,1,Information omitted,[]
7,E08,Bring three copies.,Bring 3 copies.,English,0,Number format,[]
8,E09,The library closes at 6 PM.,The library closes at 6 AM.,English,1,Changed time,[]
9,E10,The application must be submitted by Friday.,Friday is the deadline for the application.,English,0,Meaning-preserving paraphrase,[]


# 9. Evaluation and Performance Results

In [75]:

# MEANINGGUARD — FINAL HOLDOUT EVALUATION

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

evaluation_results = []

for _, row in holdout_df.iterrows():

    result = meaningguard_analyze_v7(
        original=row["original"],
        translated=row["translated"],
        target_language=row["target_language"],
        required_facts=row["required_facts"]
    )

    # 0 = no checklist issue
    # 1 = potential change or human review
    predicted = (
        0 if result["decision"] == "NO CHECKLIST ISSUE FOUND - NOT VERIFIED"
        else 1
    )

    evaluation_results.append({
        "id": row["id"],
        "language": row["target_language"],
        "case_type": row["case_type"],
        "expected": row["expected_label"],
        "predicted": predicted,
        "decision": result["decision"],
        "reasons": "; ".join(result["reasons"])
    })

results_df = pd.DataFrame(evaluation_results)

# Overall metrics
y_true = results_df["expected"]
y_pred = results_df["predicted"]

print("=" * 50)
print("MEANINGGUARD — OVERALL EVALUATION")
print("=" * 50)

print("Total examples:", len(results_df))
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))

print("\nCONFUSION MATRIX")
print("Rows = actual, columns = predicted")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["No checklist issue", "Flagged for review"],
    zero_division=0
))

# Per-language metrics
print("\n" + "=" * 50)
print("LANGUAGE-WISE PERFORMANCE")
print("=" * 50)

for language in ["English", "Hindi", "Tamil"]:

    subset = results_df[results_df["language"] == language]

    print(f"\n{language} ({len(subset)} examples)")
    print("Accuracy:", round(
        accuracy_score(subset["expected"], subset["predicted"]), 4
    ))
    print("Precision:", round(
        precision_score(subset["expected"], subset["predicted"], zero_division=0), 4
    ))
    print("Recall:", round(
        recall_score(subset["expected"], subset["predicted"], zero_division=0), 4
    ))
    print("F1:", round(
        f1_score(subset["expected"], subset["predicted"], zero_division=0), 4
    ))

print("\n" + "=" * 50)
print("CASE-BY-CASE RESULTS")
print("=" * 50)

display(results_df)

print("\nHOLDOUT EVALUATION COMPLETED")


MEANINGGUARD — OVERALL EVALUATION
Total examples: 30
Accuracy: 0.9333
Precision: 0.9333
Recall: 0.9333
F1-score: 0.9333

CONFUSION MATRIX
Rows = actual, columns = predicted
[[14  1]
 [ 1 14]]

CLASSIFICATION REPORT
                    precision    recall  f1-score   support

No checklist issue       0.93      0.93      0.93        15
Flagged for review       0.93      0.93      0.93        15

          accuracy                           0.93        30
         macro avg       0.93      0.93      0.93        30
      weighted avg       0.93      0.93      0.93        30


LANGUAGE-WISE PERFORMANCE

English (10 examples)
Accuracy: 0.9
Precision: 1.0
Recall: 0.8
F1: 0.8889

Hindi (10 examples)
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0

Tamil (10 examples)
Accuracy: 0.9
Precision: 0.8333
Recall: 1.0
F1: 0.9091

CASE-BY-CASE RESULTS


,id,language,case_type,expected,predicted,decision,reasons
0,E01,English,Equivalent time,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,
1,E02,English,Changed number,1,1,POTENTIAL MEANING CHANGE - VERIFY,Numerical details differ; Strong transformer c...
2,E03,English,Negation paraphrase,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,
3,E04,English,Negation changed,1,1,POTENTIAL MEANING CHANGE - VERIFY,Possible instruction or negation difference; S...
4,E05,English,Quantity meaning changed,1,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,
5,E06,English,Reordered information,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,
6,E07,English,Information omitted,1,1,HUMAN REVIEW REQUIRED,Possible information omission signal
7,E08,English,Number format,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,
8,E09,English,Changed time,1,1,POTENTIAL MEANING CHANGE - VERIFY,Possible time change; Strong transformer contr...
9,E10,English,Meaning-preserving paraphrase,0,0,NO CHECKLIST ISSUE FOUND - NOT VERIFIED,



HOLDOUT EVALUATION COMPLETED


# 10. Exporting Evaluation Evidence

In [76]:

# MEANINGGUARD — SAVE EVALUATION REPORT

import json

results_df.to_csv(
    "/content/MeaningGuard_Final_Evaluation.csv",
    index=False
)

summary = {
    "model": "MeaningGuard V7",
    "test_examples": 30,
    "overall_accuracy": 0.9333,
    "precision": 0.9333,
    "recall": 0.9333,
    "f1_score": 0.9333,
    "confusion_matrix": [[14, 1], [1, 14]],
    "language_accuracy": {
        "English": 0.90,
        "Hindi": 1.00,
        "Tamil": 0.90
    },
    "limitations": [
        "English quantity comparisons need improvement",
        "Tamil phrase coverage needs improvement",
        "Independent language validation is still required",
        "Small evaluation dataset"
    ]
}

with open(
    "/content/MeaningGuard_Evaluation_Summary.json",
    "w"
) as file:
    json.dump(summary, file, indent=4)

print("MeaningGuard evaluation report saved!")


MeaningGuard evaluation report saved!


# 11. Live Demonstration

In [77]:

# MEANINGGUARD — FINAL LIVE DEMONSTRATION

demo_cases = [
    {
        "name": "English — Changed time",
        "original": "The library closes at 6 PM.",
        "translated": "The library closes at 6 AM.",
        "language": "English"
    },
    {
        "name": "Hindi — Missing requirement",
        "original": "Students must bring their identity card and certificate.",
        "translated": "विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य है।",
        "language": "Hindi"
    },
    {
        "name": "Tamil — Changed time",
        "original": "The meeting starts at 4 PM.",
        "translated": "கூட்டம் மாலை 5 மணிக்கு தொடங்கும்.",
        "language": "Tamil"
    }
]

print("=" * 60)
print("             MEANINGGUARD — LIVE DEMONSTRATION")
print("=" * 60)

for case in demo_cases:

    result = meaningguard_analyze_v7(
        original=case["original"],
        translated=case["translated"],
        target_language=case["language"],
        required_facts=[]
    )

    print("\n" + "-" * 60)
    print(case["name"])
    print("-" * 60)

    print("Original:", case["original"])
    print("Translation:", case["translated"])
    print("Decision:", result["decision"])

    if result["reasons"]:
        print("Evidence:")
        for reason in result["reasons"]:
            print(" -", reason)
    else:
        print("Evidence: No checklist warning.")

print("\n" + "=" * 60)
print("DEMONSTRATION COMPLETED")
print("=" * 60)


             MEANINGGUARD — LIVE DEMONSTRATION

------------------------------------------------------------
English — Changed time
------------------------------------------------------------
Original: The library closes at 6 PM.
Translation: The library closes at 6 AM.
Decision: POTENTIAL MEANING CHANGE - VERIFY
Evidence:
 - Possible time change
 - Strong transformer contradiction signal

------------------------------------------------------------
Hindi — Missing requirement
------------------------------------------------------------
Original: Students must bring their identity card and certificate.
Translation: विद्यार्थियों को अपना पहचान पत्र लाना अनिवार्य है।
Decision: HUMAN REVIEW REQUIRED
Evidence:
 - Expected source facts require review: Certificate requirement

------------------------------------------------------------
Tamil — Changed time
------------------------------------------------------------
Original: The meeting starts at 4 PM.
Translation: கூட்டம் மாலை 5 மணிக்கு 

# 12. Final Project Evidence Package

In [78]:

# MEANINGGUARD — FINAL PROJECT PACKAGE

import zipfile
import os

files_to_package = [
    "/content/MeaningGuard_Final_Evaluation.csv",
    "/content/MeaningGuard_Evaluation_Summary.json"
]

package_path = "/content/MeaningGuard_Final_Project_Evidence.zip"

with zipfile.ZipFile(package_path, "w") as archive:
    for path in files_to_package:
        if os.path.exists(path):
            archive.write(path, arcname=os.path.basename(path))

print("Final evidence package created!")
print("Package:", package_path)

from google.colab import files
files.download(package_path)


Final evidence package created!
Package: /content/MeaningGuard_Final_Project_Evidence.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## Evaluation Summary

The prototype was evaluated on 30 candidate examples:
- English: 10
- Hindi: 10
- Tamil: 10

Preliminary results:
- Accuracy: 93.33%
- Precision: 93.33%
- Recall: 93.33%
- F1-score: 93.33%

Two cases were misclassified.

These results represent a small, controlled candidate dataset and should not be interpreted as real-world accuracy. Independent fluent-speaker validation and larger-scale testing are necessary.



# 13. Limitations and Future Improvements

- The current evaluation uses a small set of 30 candidate examples.
- Hindi and Tamil examples and labels require independent fluent-speaker validation.
- Quantity comparisons such as "at least" versus "more than" need stronger semantic handling.
- Tamil phrase coverage needs improvement to reduce false alarms.
- The system is a preliminary verification aid, not a replacement for human translation review.

## Future Improvements

- Expand and independently validate multilingual datasets.
- Improve semantic understanding of numbers, negation, and paraphrases.
- Evaluate on larger, unseen real-world translation examples.
- Improve language coverage and explainability.
